# NeuroStore: GPU/TPU Benchmark & Simulation Lab (v2: reviewed and extended)

**Purpose.** Evaluate the computational ideas behind *NeuroStore: Empirical Evaluation of a Neural-Coding Data
Architecture* on whatever accelerator Colab assigns, and say honestly how much each measurement can support.

This is an **independent, conceptual implementation**. It does not execute the report's original `coding.py`,
`memory_sdm.py`, `memory_vsa.py`, `memory_hopfield.py` or `benchmarks.py`, so nothing here should be described as
reproducing the report until sources, seeds, datasets and configurations have been reconciled.

### What changed relative to v1 (full review in `colab/REVIEW.md`)

| Problem found in v1 (each one measured, see `colab/review_evidence.json`) | What v2 does |
|---|---|
| OOD AUROC = 1.0 and 14/14 fault conditions at recall 1.0: the tests could not fail. Noisy recall at ≤ 25 % flips has an error probability near 1e-11. | Operating points chosen from an exact analytic model so baselines sit near 95 % recall; harder OOD families; many more fault models. |
| One draw of 128 queries, no intervals. The published Hopfield-vs-exact gap was not significant (McNemar p = 0.45–0.69) yet in 300 independent batches exact search wins by about 1.5 points. | Several seeds, Wilson intervals, paired exact McNemar tests, analytic cross-check. |
| p95/p99 reported from 8 samples; an 8-sample p99 underestimates the true p99 by about 27 %. | Percentiles only when enough samples exist; separate single-request latency study with hundreds of samples. |
| Sizes (N ≤ 2048, 128 queries) are launch-bound on accelerators: the arithmetic takes microseconds. | Scaling sweeps in N and batch size with achieved TFLOP/s and GB/s. |
| Global `highest` matmul precision (multi-pass on TPU, no TF32 on GPU) although ±1 data is exact in bf16/int8. | Explicit precision and kernel variants, including int8 and bit-packed popcount. |
| Results list not idempotent (re-running a cell duplicated rows), CSV lacked profile/seed/version columns. | Keyed records, full provenance columns, config hash. |
| VSA top-1 equals its oracle by construction; SDM compared without a memory-efficiency view. | Tuple-level and decode-capacity metrics validated against theory; SDM with K/M sweeps and bytes per reliable pattern. |

### Experiments

1. Noisy-key retrieval: exact flat search vs Modern-Hopfield cleanup, with analytic validation and paired statistics.
2. Hopfield deep dive: inverse temperature, iteration, denoising error, capacity.
3. Kernel and precision variants: f32 / bf16 / int8 / bit-packed popcount / approximate top-k.
4. Systems scaling: N sweep, batch sweep, single-request latency distribution, compile cost, device memory.
5. FlyHash-style sparse projection vs SimHash vs exact, with code-size accounting.
6. Role-filler VSA: record retrieval and bundle-decode capacity vs theory.
7. Sparse distributed memory: capacity, K/M sweeps, dense vs scatter/gather implementation, bytes per pattern.
8. Out-of-distribution rejection: random, chimera and partial-overlap families, several scores.
9. Fault injection: bit flips, stuck-at, dead rows/columns, variation, read noise, ADC quantisation.
10. Multi-device scaling (data parallel and memory sharded) when more than one device is present.
11. The NeuroStore Python prototype's core read kernel (sparse accumulate) on the accelerator.
12. Self-checks, summary tables, plots; 13. export and cross-hardware comparison.

### How to run

1. **Runtime → Change runtime type**: choose CPU, a GPU or a TPU. Run the notebook once per runtime you want to compare.
2. Run all cells in order. Start with `PROFILE = "quick"`. `"smoke"` is a tiny profile for checking the notebook
   itself; `"extended"` takes much longer and uses more memory.
3. Download the ZIP from the last section and use the comparison cell to merge runs.

### Rules for reading the results

* Each run benchmarks **one JAX device**, even when a TPU exposes several cores (Section 10 adds explicit multi-device runs).
* Device-resident timings exclude host transfers and index construction; transfer-inclusive latency is reported separately.
* Batch throughput is not single-request latency; both are measured and labelled.
* FlyHash, VSA and SDM are **algorithmic proxies**; SDM uses fixed-K activation, not the report's exact variant.
* Fault injection is **functional**, not a calibrated FeNAND / crossbar / compute-in-memory device model. Energy,
  endurance, ADC/DAC power and area are **not measured** here.
* Every claim should be backed by an interval or a test; the final cell prints a claims ledger with the evidence.

References: [Colab FAQ](https://research.google.com/colaboratory/faq.html),
[JAX installation](https://docs.jax.dev/en/latest/installation.html),
[JAX benchmarking](https://docs.jax.dev/en/latest/benchmarking.html).

In [ ]:
#@title Benchmark configuration (run first)
PROFILE = "quick" #@param ["smoke", "quick", "extended"]
SEED = 2026 #@param {type:"integer"}
D = 512 #@param {type:"integer"}
BETA = 16.0 #@param {type:"number"}
SIMULATE_CPU_DEVICES = 0 #@param {type:"integer"}
OUTPUT_ROOT = "" #@param {type:"string"}

RUN_HOPFIELD = True #@param {type:"boolean"}
RUN_KERNELS = True #@param {type:"boolean"}
RUN_SCALING = True #@param {type:"boolean"}
RUN_FLY = True #@param {type:"boolean"}
RUN_VSA = True #@param {type:"boolean"}
RUN_SDM = True #@param {type:"boolean"}
RUN_OOD = True #@param {type:"boolean"}
RUN_FAULTS = True #@param {type:"boolean"}
RUN_MULTIDEVICE = True #@param {type:"boolean"}
RUN_SPARSE = True #@param {type:"boolean"}

# `smoke` only checks that the notebook runs; `quick` is meant for a Colab session of a few minutes;
# `extended` uses more seeds, larger N, and long latency samples. nmax_* cap the largest stored-pattern count N
# (CPU runtimes are far smaller). budget_s stops a single sweep early (the cut-off is reported, not silent).
PROFILES = {
    "smoke": dict(seeds=2, Q=64, n_values=[64, 256], noise_levels=[0.0, 0.25, 0.40], repeats=4, warmup=1, lat_samples=40,
                  nmax_cpu=2 ** 11, nmax_acc=2 ** 11, q_sweep=[1, 8, 64], hop_betas=[1, 16, 256], hop_steps=[1, 2],
                  hop_cap_n=[64, 256, 1024], fly_noise=[0.1, 0.3, 0.4], vsa_dims=[1024], vsa_records=64, vsa_ks=[1, 5, 15],
                  vsa_trials=100, sdm_dim=128, sdm_M=[256], sdm_K=[8], sdm_loads=[10, 20, 40], sdm_noise=[0.0, 0.1],
                  ood_n=128, ood_q=128, fault_n=128, fault_q=128, sparse_N=[256, 1024], sparse_B=16, budget_s=10),
    "quick": dict(seeds=3, Q=256, n_values=[128, 512, 2048], noise_levels=[0.0, 0.10, 0.25, 0.30, 0.35, 0.40, 0.45],
                  repeats=20, warmup=3, lat_samples=300, nmax_cpu=2 ** 15, nmax_acc=2 ** 18,
                  q_sweep=[1, 4, 16, 64, 256, 1024, 4096], hop_betas=[1, 2, 4, 8, 16, 32, 64, 128, 256], hop_steps=[1, 2, 3],
                  hop_cap_n=[128, 512, 2048, 8192], fly_noise=[0.0, 0.10, 0.20, 0.25, 0.30, 0.35, 0.40],
                  vsa_dims=[1024, 2048, 4096], vsa_records=160, vsa_ks=[1, 3, 5, 9, 15, 25, 41], vsa_trials=400,
                  sdm_dim=256, sdm_M=[1024, 4096], sdm_K=[8, 16, 32], sdm_loads=[25, 50, 100, 200, 400, 800],
                  sdm_noise=[0.0, 0.10, 0.20], ood_n=512, ood_q=512, fault_n=512, fault_q=512,
                  sparse_N=[1024, 4096, 16384, 65536], sparse_B=256, budget_s=90),
    "extended": dict(seeds=5, Q=512, n_values=[128, 512, 2048, 8192], noise_levels=[0.0, 0.10, 0.25, 0.30, 0.35, 0.40, 0.45],
                     repeats=50, warmup=5, lat_samples=2000, nmax_cpu=2 ** 17, nmax_acc=2 ** 20,
                     q_sweep=[1, 4, 16, 64, 256, 1024, 4096, 16384], hop_betas=[1, 2, 4, 8, 16, 32, 64, 128, 256, 512],
                     hop_steps=[1, 2, 3, 5], hop_cap_n=[128, 512, 2048, 8192, 32768],
                     fly_noise=[0.0, 0.10, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45],
                     vsa_dims=[1024, 2048, 4096, 8192], vsa_records=640, vsa_ks=[1, 3, 5, 9, 15, 25, 41, 61], vsa_trials=1000,
                     sdm_dim=256, sdm_M=[1024, 4096, 16384], sdm_K=[8, 16, 32, 64], sdm_loads=[25, 50, 100, 200, 400, 800, 1600],
                     sdm_noise=[0.0, 0.10, 0.20], ood_n=1024, ood_q=1024, fault_n=1024, fault_q=1024,
                     sparse_N=[1024, 4096, 16384, 65536, 262144], sparse_B=512, budget_s=300),
}
assert PROFILE in PROFILES, f"unknown PROFILE {PROFILE!r}"
assert D % 32 == 0, "D must be a multiple of 32 (bit-packed kernels)"
P = PROFILES[PROFILE]
print(f"profile={PROFILE} seed={SEED} D={D} beta={BETA} seeds/profile={P['seeds']} Q={P['Q']} N={P['n_values']}")

## A. NumPy-only helpers: statistics and analytic references

Everything in this cell is pure NumPy/`math` so it can be unit-tested without JAX (see `tests/test_colab_helpers.py`).

* **Wilson / bootstrap intervals** and an **exact McNemar test** replace bare point estimates. A recall measured on
  128 queries has a 95% interval of roughly ±5 percentage points, so differences of 1–3 points between two methods
  are noise unless a paired test says otherwise.
* **`analytic_flat_recall(D, N, p)`** is the exact probability that exact nearest-neighbour search finds the target
  when the query is the target with each bit flipped with probability `p`, among `N` random ±1 patterns of
  dimension `D`. The benchmark's measured recall must agree with it; this validates data generation, noise
  injection and the retrieval kernels in a single check.
* The same formula covers VSA bundle decoding (a majority bundle of `k` terms behaves like a query with
  effective flip probability `(1 - c_k) / 2`).

In [ ]:
import math
import numpy as np

NAN = float("nan")


def wilson_ci(k, n, z=1.96):
    """Wilson score interval for a binomial proportion k/n."""
    if n <= 0:
        return (NAN, NAN)
    p = k / n
    z2 = z * z
    denom = 1.0 + z2 / n
    center = (p + z2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z2 / (4 * n * n)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


def mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value from discordant counts (b: A right/B wrong, c: A wrong/B right)."""
    n = int(b) + int(c)
    if n == 0:
        return 1.0
    k = min(int(b), int(c))
    logs = [math.lgamma(n + 1) - math.lgamma(i + 1) - math.lgamma(n - i + 1) - n * math.log(2.0) for i in range(k + 1)]
    return float(min(1.0, 2.0 * sum(math.exp(v) for v in logs)))


def paired_diff_ci(a_correct, b_correct, z=1.96):
    """Mean of (a - b) over paired 0/1 outcomes with a normal-approximation interval."""
    d = np.asarray(a_correct, float) - np.asarray(b_correct, float)
    n = len(d)
    if n < 2:
        return (NAN, NAN, NAN)
    m = float(d.mean())
    se = float(d.std(ddof=1) / math.sqrt(n))
    return (m, m - z * se, m + z * se)


def bootstrap_ci(x, stat=np.mean, n_boot=1000, alpha=0.05, seed=0):
    """Percentile bootstrap interval of `stat` over the rows of x."""
    x = np.asarray(x)
    if len(x) == 0:
        return (NAN, NAN)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    if stat is np.mean:
        s = x[idx].mean(axis=1)
    else:
        s = np.array([stat(x[i]) for i in idx])
    lo, hi = np.quantile(s, [alpha / 2, 1 - alpha / 2])
    return (float(lo), float(hi))


def rank_average(x):
    """1-based ranks with ties replaced by their average rank."""
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    xs = x[order]
    _, inv, counts = np.unique(xs, return_inverse=True, return_counts=True)
    avg = np.cumsum(counts) - (counts - 1) / 2.0
    ranks = np.empty(len(x))
    ranks[order] = avg[inv]
    return ranks


def auroc(pos, neg):
    """P(score_pos > score_neg) + 0.5 P(tie), via rank sums (O(n log n), tie aware)."""
    pos = np.asarray(pos, float)
    neg = np.asarray(neg, float)
    if len(pos) == 0 or len(neg) == 0:
        return NAN
    r = rank_average(np.concatenate([pos, neg]))
    return float((r[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2.0) / (len(pos) * len(neg)))


def tpr_at_threshold(pos, thr):
    return float(np.mean(np.asarray(pos) >= thr))


def _binom_pmf(n, p):
    k = np.arange(n + 1)
    if p <= 0.0:
        out = np.zeros(n + 1); out[0] = 1.0; return out
    if p >= 1.0:
        out = np.zeros(n + 1); out[n] = 1.0; return out
    lg = np.array([math.lgamma(i + 1) for i in range(n + 1)])
    return np.exp(lg[n] - lg[k] - lg[n - k] + k * math.log(p) + (n - k) * math.log1p(-p))


def analytic_flat_recall(D, N, p_flip):
    """Exact P(top-1 correct) of flat dot-product search.

    Target query = stored pattern with each of D bits flipped independently with probability p_flip; the other N-1
    stored patterns are independent random +-1 vectors, so their agreement count with the query is Bin(D, 1/2).
    Ties are broken uniformly at random, which equals the expectation of lowest-index tie-breaking when the target
    index is uniform. Uses E[1/(1+Bin(n,q))] = (1-(1-q)^(n+1)) / ((n+1) q).
    """
    D, N = int(D), int(N)
    pmf_t = _binom_pmf(D, 1.0 - float(p_flip))
    pmf_o = _binom_pmf(D, 0.5)
    cdf_o = np.cumsum(pmf_o)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(cdf_o > 0, pmf_o / cdf_o, 0.0)
        log_all_le = (N - 1) * np.log(cdf_o)
        tie = np.where(q > 1e-300, -np.expm1(N * np.log1p(-q)) / (N * q), 1.0)
    pc = np.where(cdf_o > 0, np.exp(log_all_le) * tie, 0.0)
    return float(np.sum(pmf_t * pc))


def solve_noise_for_recall(D, N, target, tol=1e-4):
    """Flip probability in [0, 0.5] at which analytic flat recall equals `target` (recall decreases with noise)."""
    lo, hi = 0.0, 0.5
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if analytic_flat_recall(D, N, mid) > target:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return 0.5 * (lo + hi)


def vsa_bundle_corr(k):
    """Correlation between a +-1 majority bundle of k (odd) random terms and any one of its terms.

    P(bundle bit == term bit) = 1/2 + 1/2 P(sum of the other k-1 terms is zero), so the correlation equals
    P(sum of k-1 coin flips is zero) = C(k-1, (k-1)/2) / 2^(k-1).
    """
    if k % 2 != 1:
        raise ValueError("k must be odd")
    return math.comb(k - 1, (k - 1) // 2) / 2.0 ** (k - 1)


def analytic_vsa_decode(D, n_fillers, k):
    """Expected accuracy of decoding one filler from a k-term bundle by nearest-codebook cleanup."""
    return analytic_flat_recall(D, n_fillers, (1.0 - vsa_bundle_corr(k)) / 2.0)


def ideal_sparse_code_bytes(m, k):
    """Information-theoretic size of a k-of-m binary code: log2 C(m, k) bits."""
    return (math.lgamma(m + 1) - math.lgamma(k + 1) - math.lgamma(m - k + 1)) / math.log(2.0) / 8.0


def loglog_slope(x, y):
    x, y = np.log(np.asarray(x, float)), np.log(np.asarray(y, float))
    if len(x) < 2:
        return NAN
    return float(np.polyfit(x, y, 1)[0])

In [ ]:
#@title Setup: JAX backend, devices, environment, provenance
import os, sys, json, time, gc, platform, subprocess, warnings, shutil, hashlib, datetime, importlib.metadata
from pathlib import Path
from functools import partial
from collections import Counter

if SIMULATE_CPU_DEVICES and int(SIMULATE_CPU_DEVICES) > 1:      # must happen before jax is imported
    _flag = f"--xla_force_host_platform_device_count={int(SIMULATE_CPU_DEVICES)}"
    if _flag not in os.environ.get("XLA_FLAGS", ""):
        os.environ["XLA_FLAGS"] = (os.environ.get("XLA_FLAGS", "") + " " + _flag).strip()

import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except Exception:                                              # plain Python
    display = print

try:
    import jax
    import jax.numpy as jnp
    from jax import lax
except Exception as exc:
    raise RuntimeError("JAX could not be imported. See https://docs.jax.dev/en/latest/installation.html and install the "
                       "backend matching your Colab runtime, then restart the runtime.") from exc

HI = lax.Precision.HIGHEST        # explicit per-kernel precision; v2 sets NO global matmul precision
DEFAULT_PREC = lax.Precision.DEFAULT
DEVICES = jax.local_devices()
DEVICE = DEVICES[0]
BACKEND = DEVICE.platform
DEVICE_KIND = getattr(DEVICE, "device_kind", str(DEVICE))
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def _ver(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return "n/a"


def _cmd(cmd):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else ""
    except Exception:
        return ""


def device_memory():
    """(bytes_in_use, peak_bytes_in_use, bytes_limit) of the benchmark device, None where the backend does not report."""
    try:
        s = DEVICE.memory_stats() or {}
    except Exception:
        s = {}
    return s.get("bytes_in_use"), s.get("peak_bytes_in_use"), s.get("bytes_limit")


def _host_ram():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    return int(line.split()[1]) * 1024
    except Exception:
        pass
    return None


_lim = device_memory()[2]
MEM_LIMIT = _lim if _lim else (_host_ram() if BACKEND == "cpu" else None)       # bytes available to this benchmark
NMAX = P["nmax_cpu"] if BACKEND == "cpu" else P["nmax_acc"]
CONFIG = {"profile": PROFILE, "seed": SEED, "D": D, "beta": BETA, "profile_params": P,
          "run": {k: v for k, v in globals().items() if k.startswith("RUN_") and isinstance(v, bool)}}
CONFIG_HASH = hashlib.sha1(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:10]
OUTPUT_DIR = Path(OUTPUT_ROOT or ("/content/neurostore_results" if IN_COLAB else "./neurostore_results")) / RUN_ID
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

METADATA = {
    "notebook": "NeuroStore_Colab_Benchmarks_v2", "run_id": RUN_ID, "profile": PROFILE, "seed": SEED, "config_hash": CONFIG_HASH,
    "backend": BACKEND, "device_kind": DEVICE_KIND, "local_device_count": len(DEVICES), "devices": [str(x) for x in DEVICES[:16]],
    "jax": jax.__version__, "jaxlib": _ver("jaxlib"), "numpy": np.__version__, "pandas": pd.__version__,
    "matplotlib": _ver("matplotlib"), "scipy": _ver("scipy"), "python": sys.version.split()[0], "platform": platform.platform(),
    "in_colab": IN_COLAB, "nvidia_smi": _cmd(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]),
    "tpu_environment": {k: v for k, v in os.environ.items() if k.startswith("TPU_") or k in ("COLAB_TPU_ADDR",)},
    "device_memory_limit_bytes": MEM_LIMIT, "default_matmul_precision_config": str(getattr(jax.config, "jax_default_matmul_precision", None)),
    "x64_enabled": bool(jax.config.jax_enable_x64), "largest_N_for_sweeps": NMAX,
}
META_ROW = {"run_id": RUN_ID, "profile": PROFILE, "seed_base": SEED, "config_hash": CONFIG_HASH, "backend": BACKEND,
            "device_kind": DEVICE_KIND, "jax_version": jax.__version__}
print(json.dumps(METADATA, indent=1, default=str))
if BACKEND == "cpu":
    warnings.warn("JAX is using the CPU. If you chose a GPU/TPU runtime, check the accelerator and the JAX backend installation.")
print(f"\nBenchmarks target ONE device ({DEVICE_KIND}) unless a section says otherwise; {len(DEVICES)} local device(s) visible.")

In [ ]:
#@title Infrastructure: RNG, data, timing, recording, self-checks

# ---- reproducible randomness: one independent stream per (SEED, tags...); tags are integers
def rng_for(*tags):
    return np.random.default_rng(np.random.SeedSequence([int(SEED)] + [int(t) for t in tags]))


def bipolar_i8(n, d, rng):
    """n x d random +-1 patterns as int8 (1 byte/element on the host; v1's int64 temporary needed 8)."""
    a = rng.integers(0, 2, size=(n, d), dtype=np.int8)
    a *= 2
    a -= 1
    return a


def flip_i8(x, p, rng):
    """Flip each +-1 entry independently with probability p."""
    out = x.copy()
    if p > 0:
        out[rng.random(x.shape, dtype=np.float32) < p] *= -1
    return out


def pack_bits(x_i8):
    """+-1 int8 (n, d) -> uint32 (n, d/32); popcount(a xor b) is the Hamming distance."""
    packed = np.packbits((x_i8 > 0).astype(np.uint8), axis=1)
    return np.ascontiguousarray(packed).view(np.uint32)


def put(x):
    return jax.device_put(np.asarray(x), DEVICE)


@jax.jit
def to_f32(x):
    return x.astype(jnp.float32)


@jax.jit
def to_bf16(x):
    return x.astype(jnp.bfloat16)


def nbytes(x):
    return int(x.size) * int(x.dtype.itemsize)


def q_cap(n, frac=8):
    """Largest query batch whose Q x n float32 score matrix stays below ~1/frac of memory (and 2 GiB)."""
    lim = min((MEM_LIMIT // frac) if MEM_LIMIT else 2 ** 30, 2 * 2 ** 30)
    return max(1, int(lim // (4 * max(n, 1))))


def fits(n, bytes_per_row, frac=0.4):
    return MEM_LIMIT is None or n * bytes_per_row <= frac * MEM_LIMIT


class Budget:
    """Wall-clock budget for one sweep. A sweep that hits it stops and says so (never silently)."""
    def __init__(self, seconds):
        self.t0, self.s = time.perf_counter(), float(seconds)

    def expired(self):
        return time.perf_counter() - self.t0 > self.s

    def used(self):
        return time.perf_counter() - self.t0


# ---- timing -------------------------------------------------------------------------------------------------------
def synchronize(x):
    for leaf in jax.tree_util.tree_leaves(x):
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return x


def _pct(a, q, min_n):
    return float(np.percentile(a, q)) if len(a) >= min_n else NAN


def bench(fn, *args, batch=1, samples=None, warmup=None, modes=("latency", "pipelined"), min_sample_ms=4.0, max_inner=500, keep=None):
    """Time fn(*args) on the device.

    first_call_ms     first invocation, includes tracing + compilation when the function is cold
    lat_*             one call per sample, blocked until the result is ready (what a synchronous caller sees)
    pipe_*            `inner` calls dispatched back to back, blocked once, divided by `inner` (device throughput:
                      hides per-call dispatch cost; `inner` is chosen so a sample lasts >= min_sample_ms)
    Pass a list as `keep` to receive the raw single-call latency samples (ms).
    Tail percentiles are only reported when there are enough samples to estimate them: p90 n>=10, p95 n>=20,
    p99 n>=100, p99.9 n>=1000 (otherwise NaN). With 8 samples v1's "p99" was just the maximum.
    """
    samples = P["repeats"] if samples is None else int(samples)
    warmup = P["warmup"] if warmup is None else int(warmup)
    t = time.perf_counter(); synchronize(fn(*args)); first_ms = (time.perf_counter() - t) * 1e3
    for _ in range(warmup):
        synchronize(fn(*args))
    t = time.perf_counter(); synchronize(fn(*args)); one = max(time.perf_counter() - t, 1e-7)
    inner = int(min(max_inner, max(1, math.ceil(min_sample_ms * 1e-3 / one))))
    out = {"first_call_ms": first_ms, "n_samples": samples, "inner": inner}
    gc_on = gc.isenabled()
    gc.disable()
    try:
        if "latency" in modes:
            lat = np.empty(samples)
            for i in range(samples):
                t = time.perf_counter(); synchronize(fn(*args)); lat[i] = (time.perf_counter() - t) * 1e3
            if keep is not None:
                keep.extend(lat.tolist())
            out.update(lat_min_ms=lat.min(), lat_p50_ms=np.percentile(lat, 50), lat_mean_ms=lat.mean(),
                       lat_std_ms=lat.std(ddof=1) if samples > 1 else NAN, lat_p90_ms=_pct(lat, 90, 10), lat_p95_ms=_pct(lat, 95, 20),
                       lat_p99_ms=_pct(lat, 99, 100), lat_p999_ms=_pct(lat, 99.9, 1000), lat_max_ms=lat.max())
        if "pipelined" in modes:
            pipe = np.empty(samples)
            for i in range(samples):
                t = time.perf_counter()
                outs = [fn(*args) for _ in range(inner)]
                synchronize(outs)
                pipe[i] = (time.perf_counter() - t) * 1e3 / inner
            out.update(pipe_min_ms=pipe.min(), pipe_p50_ms=np.percentile(pipe, 50), pipe_std_ms=pipe.std(ddof=1) if samples > 1 else NAN)
    finally:
        if gc_on:
            gc.enable()
    ref = out.get("pipe_p50_ms", out.get("lat_p50_ms"))
    out["qps"] = batch * 1e3 / ref
    out["us_per_query"] = ref * 1e3 / batch
    return {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in out.items()}


# ---- recording (idempotent: re-running a cell replaces its rows instead of duplicating them) ----------------------
RESULTS = {}
_KEYS = ("variant", "n", "d", "query_batch", "noise", "seed", "fault_mode", "severity", "scenario", "path", "metric")


def _norm(v):
    if isinstance(v, (float, np.floating)):
        return None if math.isnan(v) else float(v)
    if isinstance(v, np.integer):
        return int(v)
    return v


def record(experiment, method, **fields):
    row = dict(META_ROW)
    row.update(experiment=experiment, method=method)
    row.update({k: _norm(v) if not isinstance(v, (list, tuple, dict)) else v for k, v in fields.items()})
    RESULTS[(experiment, method) + tuple(_norm(fields.get(k)) for k in _KEYS)] = row
    return row


def clear(experiment):
    for k in [k for k in RESULTS if k[0] == experiment]:
        del RESULTS[k]


def results_df(experiment=None, **flt):
    df = pd.DataFrame(list(RESULTS.values()))
    if df.empty:
        return df
    if experiment is not None:
        df = df[df.experiment == experiment]
    for k, v in flt.items():
        df = df[df[k] == v]
    return df.reset_index(drop=True)


def show(experiment, cols, sort=None, max_rows=80, **flt):
    df = results_df(experiment, **flt)
    if df.empty:
        print(f"(no rows for {experiment})")
        return df
    cols = [c for c in cols if c in df.columns]
    df = df[cols]
    if sort:
        df = df.sort_values([c for c in sort if c in df.columns])
    display(df.head(max_rows).round(4))
    if len(df) > max_rows:
        print(f"... {len(df) - max_rows} more rows in results.csv")
    return df


# ---- self-checks: invariants that must hold on EVERY backend, collected into one table at the end -----------------
CHECKS = []


def check(name, ok, detail=""):
    ok = bool(ok)
    CHECKS[:] = [c for c in CHECKS if c["name"] != name]
    CHECKS.append({"name": name, "ok": ok, "detail": str(detail)})
    print(("PASS  " if ok else "FAIL  ") + name + (f"  | {detail}" if detail else ""))
    return ok


def savefig(fig, name):
    fig.savefig(OUTPUT_DIR / "plots" / f"{name}.png", dpi=140, bbox_inches="tight")


def yerr(r, lo, hi):
    """Error-bar lengths from a point estimate and interval bounds, clipped at 0 (fp edge cases at recall 1.0)."""
    r, lo, hi = (np.asarray(v, float) for v in (r, lo, hi))
    return np.vstack([np.clip(r - lo, 0, None), np.clip(hi - r, 0, None)])


def pooled_recall(correct_list):
    """Pool boolean arrays over seeds -> (recall, lo, hi, trials)."""
    c = np.concatenate(correct_list) if len(correct_list) else np.zeros(0, bool)
    lo, hi = wilson_ci(int(c.sum()), len(c))
    return (float(c.mean()) if len(c) else NAN), lo, hi, int(len(c))


SEEDS = list(range(P["seeds"]))
print(f"infrastructure ready: {len(SEEDS)} seeds, device-memory limit = {None if MEM_LIMIT is None else round(MEM_LIMIT / 2**30, 2)} GiB")

## 1. Noisy-key associative retrieval: exact flat search vs Modern-Hopfield cleanup

Equal-norm ±1 patterns, queries are stored patterns with each bit flipped independently, **the same queries for both
methods** (paired). Results are pooled over several seeds (each seed = a new memory and new queries) and reported with
Wilson 95 % intervals. The `analytic` column is the exact expectation for flat search; measured recall must agree
with it (self-check). Rows where the analytic recall is 1 − 1e-9 or better carry no information about either method.

**How to read it.** Hopfield softmax preserves the ranking of its input similarities, so the *first* stage cannot
differ from exact search. Any difference comes from the reconstruction-and-cleanup stage (cross-talk between stored
patterns). The McNemar column tests whether the paired difference is distinguishable from zero.

In [ ]:
#@title 1. Noisy-key retrieval (multi-seed, paired, analytically validated)
clear("noisy_recall"); clear("noisy_recall_timing")


@jax.jit
def exact_eval(mem, q):
    s = jnp.matmul(q, mem.T, precision=HI)
    mx = jnp.max(s, axis=1)
    return jnp.argmax(s, axis=1), lax.top_k(s, 10)[1], jnp.sum(s == mx[:, None], axis=1)


@jax.jit
def hopfield_eval(mem, q, beta):
    s = jnp.matmul(q, mem.T, precision=HI) / mem.shape[1]
    w = jax.nn.softmax(beta * s, axis=-1)
    rec = jnp.matmul(w, mem, precision=HI)                       # attention read-out
    c = jnp.matmul(rec, mem.T, precision=HI)                     # cleanup against the stored patterns
    return jnp.argmax(c, axis=1), lax.top_k(c, 10)[1], rec


def make_queries(mem_np, n_q, noise, rng):
    tg = rng.integers(0, len(mem_np), size=n_q)
    return tg, flip_i8(mem_np[tg], noise, rng)


N_VALUES, NOISE = P["n_values"], P["noise_levels"]
S1 = {}
for n in N_VALUES:
    for si in SEEDS:
        mem_np = bipolar_i8(n, D, rng_for(1, 0, n, si))
        mem = to_f32(put(mem_np))
        for noise in NOISE:
            r = rng_for(1, 1, n, int(round(noise * 1000)), si)
            tg, q_np = make_queries(mem_np, P["Q"], noise, r)
            q = to_f32(put(q_np))
            e1, e10, ties = (np.asarray(a) for a in exact_eval(mem, q))
            h1, h10, rec = hopfield_eval(mem, q, BETA)
            h1, h10, rec = np.asarray(h1), np.asarray(h10), np.asarray(rec)
            d = S1.setdefault((n, noise), {k: [] for k in ("e1", "e10", "h1", "h10", "ties", "ber", "agree")})
            d["e1"].append(e1 == tg); d["h1"].append(h1 == tg)
            d["e10"].append((e10 == tg[:, None]).any(1)); d["h10"].append((h10 == tg[:, None]).any(1))
            d["ties"].append(ties > 1); d["agree"].append(e1 == h1)
            d["ber"].append((np.where(rec >= 0, 1, -1) != mem_np[tg]).mean(axis=1))
        if si == 0:                                   # device timing once per N (does not depend on the noise level)
            r = rng_for(1, 2, n)
            _, q_np = make_queries(mem_np, P["Q"], 0.25, r)
            q = to_f32(put(q_np))
            for name, fn, args in (("exact_flat", exact_eval, (mem, q)), ("hopfield_cleanup", hopfield_eval, (mem, q, BETA))):
                record("noisy_recall_timing", name, n=n, d=D, query_batch=P["Q"], **bench(fn, *args, batch=P["Q"]), index_bytes=nbytes(mem))

rows = []
for (n, noise), d in sorted(S1.items()):
    ana = analytic_flat_recall(D, n, noise)
    e1, h1 = np.concatenate(d["e1"]), np.concatenate(d["h1"])
    b, c = int((e1 & ~h1).sum()), int((~e1 & h1).sum())
    pd_m, pd_lo, pd_hi = paired_diff_ci(h1, e1)
    for tag, name in (("e", "exact_flat"), ("h", "hopfield_cleanup")):
        r1, lo, hi, trials = pooled_recall(d[tag + "1"])
        r10, lo10, hi10, _ = pooled_recall(d[tag + "10"])
        se = math.sqrt(max(ana * (1 - ana), 1e-12) / trials)
        extra = dict(analytic_recall=ana, z_vs_analytic=(r1 - ana) / se) if tag == "e" else dict(
            paired_diff_vs_exact=pd_m, paired_diff_lo=pd_lo, paired_diff_hi=pd_hi, mcnemar_p=mcnemar_exact(b, c),
            disagreement_with_exact=1 - float(np.concatenate(d["agree"]).mean()), ber_out=float(np.concatenate(d["ber"]).mean()), ber_in=noise)
        rows.append(record("noisy_recall", name, n=n, d=D, noise=noise, query_batch=P["Q"], trials=trials, n_seeds=len(SEEDS),
                           recall_at_1=r1, recall_at_1_lo=lo, recall_at_1_hi=hi, recall_at_10=r10, recall_at_10_lo=lo10, recall_at_10_hi=hi10,
                           tie_fraction=float(np.concatenate(d["ties"]).mean()), index_bytes=n * D * 4, **extra))

view = pd.DataFrame(rows)
tbl = view.pivot_table(index=["n", "noise"], columns="method", values=["recall_at_1", "recall_at_1_lo", "recall_at_1_hi"]).round(4)
out = pd.DataFrame({
    "trials": view[view.method == "exact_flat"].set_index(["n", "noise"]).trials,
    "exact [95% CI]": [f"{r:.3f} [{lo:.3f},{hi:.3f}]" for r, lo, hi in zip(*[view[view.method == 'exact_flat'][c] for c in ("recall_at_1", "recall_at_1_lo", "recall_at_1_hi")])],
    "hopfield [95% CI]": [f"{r:.3f} [{lo:.3f},{hi:.3f}]" for r, lo, hi in zip(*[view[view.method == 'hopfield_cleanup'][c] for c in ("recall_at_1", "recall_at_1_lo", "recall_at_1_hi")])],
    "analytic exact": view[view.method == "exact_flat"].set_index(["n", "noise"]).analytic_recall.round(5),
    "Δ hopfield−exact": view[view.method == "hopfield_cleanup"].set_index(["n", "noise"]).paired_diff_vs_exact.round(4),
    "McNemar p": view[view.method == "hopfield_cleanup"].set_index(["n", "noise"]).mcnemar_p.round(4),
    "disagree": view[view.method == "hopfield_cleanup"].set_index(["n", "noise"]).disagreement_with_exact.round(4),
})
display(out)
show("noisy_recall_timing", ["method", "n", "query_batch", "first_call_ms", "lat_p50_ms", "pipe_p50_ms", "qps", "n_samples"], sort=["n", "method"])

# ---- self-check: measured exact recall must match the analytic expectation
bad = [(n, nz, round(r.z_vs_analytic, 2)) for (n, nz), r in
       ((k, view[(view.n == k[0]) & (view.noise == k[1]) & (view.method == "exact_flat")].iloc[0]) for k in S1)
       if abs(r.recall_at_1 - r.analytic_recall) > 4.5 * math.sqrt(max(r.analytic_recall * (1 - r.analytic_recall), 1e-12) / r.trials) + 1.0 / r.trials]
check("S1 exact-search recall agrees with the analytic model (|z| <= 4.5)", not bad, f"violations: {bad}" if bad else f"{len(S1)} (N, noise) cells")
sat = view[(view.method == "exact_flat") & (view.analytic_recall > 1 - 1e-9)]
print(f"{len(sat)}/{len(S1)} (N, noise) cells have an analytic error probability below 1e-9: they cannot discriminate any method.")

fig, axes = plt.subplots(1, len(N_VALUES), figsize=(4.4 * len(N_VALUES), 3.8), sharey=True, squeeze=False)
for ax, n in zip(axes[0], N_VALUES):
    for name, mk in (("exact_flat", "o"), ("hopfield_cleanup", "s")):
        g = view[(view.n == n) & (view.method == name)].sort_values("noise")
        ax.errorbar(g.noise, g.recall_at_1, yerr=yerr(g.recall_at_1, g.recall_at_1_lo, g.recall_at_1_hi), marker=mk, capsize=2, label=name, alpha=.85)
    g = view[(view.n == n) & (view.method == "exact_flat")].sort_values("noise")
    ax.plot(g.noise, g.analytic_recall, "k--", lw=1, label="analytic (exact)")
    ax.set(title=f"N={n}, D={D}", xlabel="fraction of query bits flipped"); ax.grid(alpha=.25)
axes[0][0].set_ylabel("recall@1 (95% Wilson CI)"); axes[0][0].legend(fontsize=8)
plt.tight_layout(); savefig(fig, "s1_noisy_recall"); plt.show()

## 2. Hopfield deep dive: inverse temperature, iteration, denoising, capacity

* **β sweep.** As β → ∞ the softmax becomes one-hot and the cleanup stage reproduces exact search; small β mixes many
  patterns (cross-talk). This quantifies *how much* the Hopfield stage costs relative to exact search.
* **Iteration.** `T` steps of ξ ← sign(Mᵀ softmax(β M ξ / D)); each step can denoise the *query itself*, which exact
  index lookup cannot do. Bit-error rate (BER) is measured against the stored target pattern.
* **Capacity.** Recall vs number of stored patterns at a fixed noise level, against the exact-search analytic curve.

In [ ]:
#@title 2. Hopfield: beta sweep, iteration, denoising BER, capacity
clear("hopfield_beta"); clear("hopfield_capacity")


@partial(jax.jit, static_argnames=("steps",))
def hopfield_iter(mem, q, beta, steps):
    d = mem.shape[1]
    xi = q
    for _ in range(steps):
        w = jax.nn.softmax(beta * jnp.matmul(xi, mem.T, precision=HI) / d, axis=-1)
        xi = jnp.where(jnp.matmul(w, mem, precision=HI) >= 0, 1.0, -1.0)
    return jnp.argmax(jnp.matmul(xi, mem.T, precision=HI), axis=1), xi


if RUN_HOPFIELD:
    n_h = max(n for n in N_VALUES if n <= 2048)
    BETA_BIG = float(max(4096, 16 * D))                  # beta * (2/D) >= 32: softmax is one-hot unless scores tie exactly
    betas = list(P["hop_betas"]) + [BETA_BIG]
    acc, uniq_mismatch, uniq_total = {}, {0.30: 0, 0.40: 0}, {0.30: 0, 0.40: 0}
    for si in SEEDS:
        mem_np = bipolar_i8(n_h, D, rng_for(2, 0, n_h, si)); mem = to_f32(put(mem_np))
        for noise in (0.30, 0.40):
            r = rng_for(2, 1, n_h, int(noise * 1000), si)
            tg, q_np = make_queries(mem_np, P["Q"], noise, r)
            q = to_f32(put(q_np))
            e_idx, _, e_ties = (np.asarray(a) for a in exact_eval(mem, q))
            e1 = e_idx == tg
            acc.setdefault((noise, "exact", 0, 0.0), []).append((e1, np.zeros(len(tg))))
            for beta in betas:
                for steps in P["hop_steps"]:
                    idx, xi = hopfield_iter(mem, q, float(beta), steps)
                    ber = (np.asarray(xi) != mem_np[tg]).mean(axis=1)
                    if beta == BETA_BIG and steps == P["hop_steps"][0]:
                        u = e_ties == 1                                   # queries whose best score is unique
                        uniq_mismatch[noise] += int((np.asarray(idx)[u] != e_idx[u]).sum()); uniq_total[noise] += int(u.sum())
                    acc.setdefault((noise, "hopfield", steps, float(beta)), []).append((np.asarray(idx) == tg, ber))
    for (noise, kind, steps, beta), lst in sorted(acc.items()):
        r1, lo, hi, trials = pooled_recall([a for a, _ in lst])
        record("hopfield_beta", "exact_flat" if kind == "exact" else f"hopfield_T{steps}", n=n_h, d=D, noise=noise, query_batch=P["Q"], variant=beta,
               trials=trials, recall_at_1=r1, recall_at_1_lo=lo, recall_at_1_hi=hi, analytic_recall=analytic_flat_recall(D, n_h, noise),
               ber_out=float(np.mean(np.concatenate([b for _, b in lst]))) if kind == "hopfield" else NAN, ber_in=noise)
    hb = results_df("hopfield_beta")
    # large-beta equivalence with exact search: must be identical whenever the best score is unique
    for noise in (0.30, 0.40):
        check(f"S2 Hopfield(beta={BETA_BIG:g}) equals exact search wherever the best score is unique (noise {noise})", uniq_mismatch[noise] == 0,
              f"{uniq_mismatch[noise]} mismatches over {uniq_total[noise]} unique-max queries")
    show("hopfield_beta", ["method", "noise", "variant", "recall_at_1", "recall_at_1_lo", "recall_at_1_hi", "ber_in", "ber_out", "analytic_recall"],
         sort=["noise", "method", "variant"], max_rows=40)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.9))
    for noise, ls in ((0.30, "-"), (0.40, "--")):
        g = hb[(hb.noise == noise) & (hb.method == f"hopfield_T{P['hop_steps'][0]}")].sort_values("variant")
        axes[0].errorbar(g.variant, g.recall_at_1, yerr=yerr(g.recall_at_1, g.recall_at_1_lo, g.recall_at_1_hi), ls=ls, marker="o", capsize=2, label=f"hopfield T={P['hop_steps'][0]}, noise {noise}")
        axes[0].axhline(hb[(hb.noise == noise) & (hb.method == "exact_flat")].recall_at_1.iloc[0], color="k", ls=ls, lw=.8)
        for steps in P["hop_steps"]:
            g = hb[(hb.noise == noise) & (hb.method == f"hopfield_T{steps}") & (hb.variant == 16.0)]
            if len(g):
                axes[1].plot([steps], g.ber_out, "o" if noise == 0.30 else "s", label=f"noise {noise}" if steps == P["hop_steps"][0] else None, color="C0" if noise == 0.30 else "C1")
    axes[0].set(xscale="log", xlabel="inverse temperature beta", ylabel="recall@1", title=f"Hopfield vs exact (black), N={n_h}"); axes[0].legend(fontsize=7); axes[0].grid(alpha=.25)
    axes[1].set(xlabel="iteration steps T", ylabel="output bit-error rate (beta=16)", title="denoising of the query pattern"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.25)

    # capacity vs N
    cap_noise, cap_betas = 0.30, [4.0, 16.0, 64.0]
    cap = {}
    for n in P["hop_cap_n"]:
        if not fits(n, D * 4 * 3) or n > max(NMAX, 2048):
            print(f"capacity sweep: skipping N={n} (memory / profile limit)")
            continue
        for si in SEEDS:
            mem_np = bipolar_i8(n, D, rng_for(2, 2, n, si)); mem = to_f32(put(mem_np))
            r = rng_for(2, 3, n, si); tg, q_np = make_queries(mem_np, min(P["Q"], q_cap(n)), cap_noise, r); q = to_f32(put(q_np))
            cap.setdefault((n, "exact_flat"), []).append(np.asarray(exact_eval(mem, q)[0]) == tg)
            for beta in cap_betas:
                cap.setdefault((n, f"hopfield_beta{beta:g}"), []).append(np.asarray(hopfield_eval(mem, q, beta)[0]) == tg)
    for (n, name), lst in sorted(cap.items()):
        r1, lo, hi, trials = pooled_recall(lst)
        record("hopfield_capacity", name, n=n, d=D, noise=cap_noise, trials=trials, recall_at_1=r1, recall_at_1_lo=lo, recall_at_1_hi=hi,
               analytic_recall=analytic_flat_recall(D, n, cap_noise))
    cp = results_df("hopfield_capacity")
    for name in sorted(cp.method.unique()):
        g = cp[cp.method == name].sort_values("n")
        axes[2].errorbar(g.n, g.recall_at_1, yerr=yerr(g.recall_at_1, g.recall_at_1_lo, g.recall_at_1_hi), marker="o", capsize=2, label=name)
    g = cp[cp.method == "exact_flat"].sort_values("n")
    axes[2].plot(g.n, g.analytic_recall, "k--", lw=1, label="analytic exact")
    axes[2].set(xscale="log", xlabel="stored patterns N", ylabel="recall@1", title=f"capacity at {int(cap_noise * 100)}% flips (D={D})"); axes[2].legend(fontsize=7); axes[2].grid(alpha=.25)
    plt.tight_layout(); savefig(fig, "s2_hopfield"); plt.show()
else:
    print("Hopfield deep dive disabled.")

## 3. Kernel and precision variants for exact flat search

v1 set a global `highest` matmul precision. For ±1 data that is unnecessary: every product is ±1 and every partial
sum is an integer far below 2²⁴, so bf16 or int8 inputs with a wide accumulator give *bit-identical* scores. On
hardware where `highest` costs several passes (TPU) or disables TF32 (GPU) this changes throughput materially. This
section measures that instead of assuming it.

| variant | inputs | accumulate | notes |
|---|---|---|---|
| `f32_highest` | float32 | fp32 | v1's choice |
| `f32_default` | float32 | backend default | may use TF32 / bf16 passes depending on hardware |
| `bf16_acc32` | bfloat16 | fp32 | `preferred_element_type=float32` |
| `bf16_bf16out` | bfloat16 | rounded to bf16 | what a naive `a @ b.T` on bf16 arrays does; exact only while scores fit in 8 significant bits |
| `int8_acc32` | int8 | int32 | integer MXU / tensor-core path |
| `packed_popcount` | 1 bit/element in uint32 words | integer | `popcount(a xor b)` = Hamming distance; 32× smaller than fp32 |

**Correctness criterion.** Exact ties are common for integer scores, so "same index as the reference" is the wrong
test. A kernel is *exact* iff the pattern it picks attains the true maximum score (`optimal_rate == 1`).

In [ ]:
#@title 3. Kernel / precision variants (accuracy, throughput, footprint)
clear("kernel_variants"); clear("topk_variants"); clear("precision_vs_D")

K_F32_HI = jax.jit(lambda q, m: jnp.argmax(jnp.matmul(q, m.T, precision=HI), axis=1))
K_F32_DEF = jax.jit(lambda q, m: jnp.argmax(jnp.matmul(q, m.T, precision=DEFAULT_PREC), axis=1))
K_BF16_ACC32 = jax.jit(lambda q, m: jnp.argmax(jnp.matmul(q, m.T, preferred_element_type=jnp.float32), axis=1))
K_BF16_OUT = jax.jit(lambda q, m: jnp.argmax(jnp.matmul(q, m.T), axis=1))
K_I8 = jax.jit(lambda q, m: jnp.argmax(lax.dot_general(q, m, (((1,), (1,)), ((), ())), preferred_element_type=jnp.int32), axis=1))


@partial(jax.jit, static_argnames=("block",))
def k_popcount(qw, mw, block):
    """Top-1 by minimum Hamming distance, scanning the memory in blocks so the (Q, block, words) temporary stays small."""
    n, w = mw.shape
    nb = n // block
    mb = mw.reshape(nb, block, w)
    base = jnp.arange(nb, dtype=jnp.int32) * block

    def body(carry, xs):
        best_d, best_i = carry
        blk, b0 = xs
        dist = jnp.sum(lax.population_count(jnp.bitwise_xor(qw[:, None, :], blk[None, :, :])).astype(jnp.int32), axis=-1)
        loc = jnp.argmin(dist, axis=1)
        dm = jnp.take_along_axis(dist, loc[:, None], axis=1)[:, 0]
        better = dm < best_d                                     # strict: keeps the lowest index on ties, like argmax
        return (jnp.where(better, dm, best_d), jnp.where(better, b0 + loc, best_i)), None

    init = (jnp.full((qw.shape[0],), 2 ** 30, jnp.int32), jnp.zeros((qw.shape[0],), jnp.int32))
    (_, best_i), _ = lax.scan(body, init, (mb, base))
    return best_i


def pick_block(n, q, w, max_elems=2 ** 25):
    cap = max(64, max_elems // max(1, q * w))
    b = 1
    while b * 2 <= min(cap, n) and n % (b * 2) == 0:
        b *= 2
    return b


K_POP = lambda q, m: k_popcount(q, m, block=pick_block(m.shape[0], q.shape[0], m.shape[1]))
KERNELS = {          # name -> (callable, operand family, bytes per stored element)
    "f32_highest": (K_F32_HI, "f32", 4.0), "f32_default": (K_F32_DEF, "f32", 4.0), "bf16_acc32": (K_BF16_ACC32, "bf16", 2.0),
    "bf16_bf16out": (K_BF16_OUT, "bf16", 2.0), "int8_acc32": (K_I8, "i8", 1.0), "packed_popcount": (K_POP, "pack", 1 / 8),
}
# CPUs have no fast bf16/int8 matmul in XLA; cap their size so a CPU session does not stall (accelerators are uncapped)
CPU_NMAX = {"bf16_acc32": 2 ** 13, "bf16_bf16out": 2 ** 13, "int8_acc32": 2 ** 12}


def kernel_allowed(name, n):
    return not (BACKEND == "cpu" and n > CPU_NMAX.get(name, 10 ** 12))


def operands(mem_i8, q_i8):
    m8, q8 = put(mem_i8), put(q_i8)
    return {"f32": (to_f32(q8), to_f32(m8)), "bf16": (to_bf16(q8), to_bf16(m8)), "i8": (q8, m8),
            "pack": (put(pack_bits(q_i8)), put(pack_bits(mem_i8)))}


def ref_max_scores(q32, m32, chunk=1 << 16):
    best = None
    for s in range(0, m32.shape[0], chunk):
        blk = jnp.max(jnp.matmul(q32, m32[s:s + chunk].T, precision=HI), axis=1)
        best = blk if best is None else jnp.maximum(best, blk)
    return np.asarray(best)


def optimal_rate(pred, q_i8, mem_i8, ref_max):
    got = (q_i8.astype(np.int32) * mem_i8[np.asarray(pred)].astype(np.int32)).sum(axis=1)
    return float(np.mean(got == ref_max))


def run_kernel(experiment, name, ops, q_i8, mem_i8, rmax, tg, n, nv, noise, **bench_kw):
    """Accuracy (does the pick attain the maximal score?), timing and footprint of one kernel on prepared operands."""
    fn, fam, eb = KERNELS[name]
    qop, mop = ops[fam]
    qn = qop.shape[0]
    pred = np.asarray(fn(qop, mop))
    st = bench(fn, qop, mop, batch=qn, **bench_kw)
    t = st["pipe_p50_ms"] * 1e-3
    used, _, _ = device_memory()
    return record(experiment, name, n=n, d=D, query_batch=qn, noise=noise, optimal_rate=optimal_rate(pred[:nv], q_i8[:nv], mem_i8, rmax[:nv]),
                  recall_at_1=float(np.mean(pred == tg)), gflops_effective=2.0 * qn * n * D / t / 1e9,
                  gbps_memory=(n * D * eb + qn * D * eb) / t / 1e9, operand_bytes=nbytes(mop), device_bytes_in_use=used, **st)


def sizes_for_kernels():
    cand = [256, 1024] if PROFILE == "smoke" else [1024, 8192, 65536, 262144, 1048576]
    return [n for n in cand if n <= NMAX and fits(n, D * 7.2)]


if RUN_KERNELS:
    KN, NV, NOISE_K = sizes_for_kernels(), 128, 0.40
    for n in KN:
        Qn = min(P["Q"], q_cap(n))
        mem_i8 = bipolar_i8(n, D, rng_for(3, 0, n))
        r = rng_for(3, 1, n)
        tg, q_i8 = make_queries(mem_i8, Qn, NOISE_K, r)
        ops = operands(mem_i8, q_i8)
        nv = min(Qn, NV)
        rmax = ref_max_scores(ops["f32"][0][:nv], ops["f32"][1])
        for name in KERNELS:
            if not kernel_allowed(name, n):
                continue
            run_kernel("kernel_variants", name, ops, q_i8, mem_i8, rmax, tg, n, nv, NOISE_K)
        # exact vs approximate top-10 (matters on TPU where lax.top_k over large N is slow)
        q32, m32 = ops["f32"]
        f_ex = jax.jit(lambda q, m: lax.top_k(jnp.matmul(q, m.T, precision=HI), 10)[1])
        f_ap = jax.jit(lambda q, m: lax.approx_max_k(jnp.matmul(q, m.T, precision=HI), 10, recall_target=0.95)[1])
        ex10 = np.asarray(f_ex(q32, m32))
        for name, fn in (("top10_exact", f_ex), ("top10_approx_r0.95", f_ap)):
            try:
                got = np.asarray(fn(q32, m32))
                ov = float(np.mean([len(set(a) & set(b)) / 10 for a, b in zip(got, ex10)]))
                st = bench(fn, q32, m32, batch=Qn)
                record("topk_variants", name, n=n, d=D, query_batch=Qn, set_overlap_with_exact=ov, hit_at_10=float(np.mean((got == tg[:, None]).any(1))), **st)
            except Exception as exc:                                    # approx_max_k is backend dependent
                print(f"{name} unavailable at N={n}: {type(exc).__name__}: {str(exc)[:100]}")
        del ops, q32, m32
        gc.collect()

    kv = results_df("kernel_variants")
    show("kernel_variants", ["method", "n", "query_batch", "optimal_rate", "first_call_ms", "lat_p50_ms", "pipe_p50_ms", "qps", "gflops_effective",
                             "gbps_memory", "operand_bytes"], sort=["n", "method"])
    show("topk_variants", ["method", "n", "query_batch", "set_overlap_with_exact", "hit_at_10", "lat_p50_ms", "pipe_p50_ms"], sort=["n", "method"])

    exact_names = ["f32_highest", "f32_default", "bf16_acc32", "int8_acc32", "packed_popcount"]
    sub = kv[kv.method.isin(exact_names)]
    check("S3 f32/bf16(acc32)/int8/popcount kernels all select a maximal-score pattern", bool((sub.optimal_rate == 1.0).all()),
          f"{len(sub)} (kernel, N) cells; min optimal_rate={sub.optimal_rate.min():.4f}" if len(sub) else "no cells")
    nb = kv[kv.method == "bf16_bf16out"]
    if len(nb):
        print(f"naive bf16-output variant: optimal_rate min={nb.optimal_rate.min():.4f} at D={D} "
              f"({'exact here' if (nb.optimal_rate == 1).all() else 'NOT exact: scores rounded to bf16 create false ties'}); see precision_vs_D below.")

    # when does rounding the *output* to bf16 matter? scores are even integers up to D: exact while |score| <= 512, spaced by 2^k beyond
    for dd in sorted({512, 2048, 8192, D}):
        nn_, qq_ = 1024, 128
        mem_i8 = bipolar_i8(nn_, dd, rng_for(3, 5, dd)); r = rng_for(3, 6, dd)
        tg, q_i8 = make_queries(mem_i8, qq_, 0.30, r)
        m32, q32 = to_f32(put(mem_i8)), to_f32(put(q_i8))
        s_ref = np.asarray(jnp.matmul(q32, m32.T, precision=HI))
        s_bf = np.asarray(jnp.matmul(to_bf16(q32), to_bf16(m32).T)).astype(np.float32)
        picked = s_ref[np.arange(qq_), s_bf.argmax(1)]
        record("precision_vs_D", "bf16_bf16out", n=nn_, d=dd, query_batch=qq_, noise=0.30, optimal_rate=float(np.mean(picked == s_ref.max(1))),
               frac_scores_changed=float(np.mean(s_bf != s_ref)))
    show("precision_vs_D", ["method", "d", "n", "optimal_rate", "frac_scores_changed"], sort=["d"])

    if len(kv):
        fig, ax = plt.subplots(figsize=(7.5, 3.8))
        for name in KERNELS:
            g = kv[kv.method == name].sort_values("n")
            if len(g):
                ax.plot(g.n, g.pipe_p50_ms, marker="o", label=name)
        ax.set(xscale="log", yscale="log", xlabel="stored patterns N", ylabel="pipelined time per call (ms)", title=f"exact top-1, D={D}, {BACKEND}:{DEVICE_KIND}")
        ax.grid(alpha=.25, which="both"); ax.legend(fontsize=8); plt.tight_layout(); savefig(fig, "s3_kernels"); plt.show()
else:
    print("Kernel study disabled.")

## 4. Systems scaling: N sweep, batch sweep, single-request latency, compile cost, memory

v1's largest workload (N = 2048, 256 queries, D = 512) is about 0.5 GFLOP, which a modern accelerator finishes in
microseconds; the measured milliseconds were launch and synchronisation overhead, so v1 could not distinguish a CPU from
a GPU or TPU. This section therefore:

* measures the **dispatch floor** (an empty kernel plus a blocking read) and flags every row whose latency is within
  5× of it as *launch-bound*;
* sweeps **N** until memory or the profile budget stops it and reports achieved GFLOP/s and GB/s;
* sweeps the **batch size** (Q = 1 … thousands) to separate latency from throughput;
* measures the **single-request latency distribution** (Q = 1) with hundreds to thousands of samples, both
  device-resident and including host → device → host transfer, and only then reports p99 / p99.9;
* reports **first-call (compile) cost** and **device memory in use**.

On a CPU runtime these sweeps are only a functional check; they say nothing about accelerator behaviour.

In [ ]:
#@title 4. Scaling sweeps, batch sweep, latency distribution
clear("scaling_N"); clear("scaling_Q"); clear("latency_dist"); clear("scaling_summary"); clear("dispatch_floor")
SWEEP_KERNELS = ["f32_highest", "f32_default", "bf16_acc32", "int8_acc32", "packed_popcount"]
BATCH_KERNELS = ["f32_highest", "bf16_acc32", "int8_acc32", "packed_popcount"]
LAT_SAMPLES = {}
HOST_OPERAND = {"f32": lambda a: to_f32(put(a)), "bf16": lambda a: to_bf16(put(a)), "i8": lambda a: put(a), "pack": lambda a: put(pack_bits(a))}


def n_sweep_list():
    top = int(math.log2(NMAX))
    es = list(range(8 if PROFILE == "smoke" else 10, top + 1, 1 if PROFILE == "smoke" else 2))
    if es[-1] != top:
        es.append(top)
    return [2 ** e for e in es if fits(2 ** e, D * 7.2)]


# dispatch floor: the cost of launching an (almost) empty kernel and reading its result back
_tiny = jax.jit(lambda x: x + 1.0)
_x = put(np.ones(8, np.float32))
_fl = bench(_tiny, _x, batch=1, samples=max(200, P["lat_samples"] // 2), modes=("latency",))
DISPATCH_FLOOR_MS = _fl["lat_p50_ms"]
record("dispatch_floor", "empty_kernel", d=D, query_batch=1, **_fl)
print(f"dispatch floor (empty kernel + blocking read): p50 {DISPATCH_FLOOR_MS * 1e3:.1f} us, p99 {_fl['lat_p99_ms'] * 1e3:.1f} us "
      f"(from {_fl['n_samples']} samples)")

if RUN_SCALING:
    ns = n_sweep_list()
    big = bipolar_i8(ns[-1], D, rng_for(4, 0))
    bud = Budget(P["budget_s"])
    for n in ns:
        if bud.expired():
            print(f"N sweep stopped before N={n}: budget of {bud.s:.0f}s used")
            break
        Qn = min(P["Q"], q_cap(n))
        tg, q_i8 = make_queries(big[:n], Qn, 0.25, rng_for(4, 1, n))
        ops = operands(big[:n], q_i8)
        nv = min(Qn, 64)
        rmax = ref_max_scores(ops["f32"][0][:nv], ops["f32"][1])
        for name in SWEEP_KERNELS:
            if kernel_allowed(name, n):
                row = run_kernel("scaling_N", name, ops, q_i8, big[:n], rmax, tg, n, nv, 0.25)
                row["launch_bound"] = bool(row["lat_p50_ms"] < 5 * DISPATCH_FLOOR_MS)
        del ops
        gc.collect()
    sc = results_df("scaling_N")
    for name in SWEEP_KERNELS:
        g = sc[sc.method == name].sort_values("n")
        if len(g) >= 2:
            tail = g.tail(3)
            record("scaling_summary", name, n=int(g.n.max()), d=D, scenario="N_sweep", scaling_exponent=loglog_slope(tail.n, tail.pipe_p50_ms),
                   best_gflops=float(g.gflops_effective.max()), best_gbps=float(g.gbps_memory.max()), largest_n=int(g.n.max()),
                   pipe_ms_at_largest=float(g.pipe_p50_ms.iloc[-1]), launch_bound_rows=int((g.lat_p50_ms < 5 * DISPATCH_FLOOR_MS).sum()), rows=len(g))
    show("scaling_N", ["method", "n", "query_batch", "optimal_rate", "first_call_ms", "lat_p50_ms", "pipe_p50_ms", "gflops_effective", "gbps_memory", "device_bytes_in_use"],
         sort=["method", "n"], max_rows=60)
    show("scaling_summary", ["method", "largest_n", "scaling_exponent", "best_gflops", "best_gbps", "pipe_ms_at_largest", "launch_bound_rows", "rows"], sort=["method"])
    nbad = sc[sc.method.isin(["f32_highest", "bf16_acc32", "int8_acc32", "packed_popcount", "f32_default"])]
    check("S4 every kernel in the N sweep selects a maximal-score pattern", bool((nbad.optimal_rate == 1.0).all()), f"{len(nbad)} (kernel, N) cells")

    # ---------------- batch-size sweep at a fixed N
    nb = min(ns[-1], 2 ** 16)
    qs = [q for q in P["q_sweep"] if q <= q_cap(nb)]
    mem_i8 = big[:nb]
    tg_all, q_all = make_queries(mem_i8, max(qs), 0.25, rng_for(4, 2))
    ops_all = operands(mem_i8, q_all)
    rmax32 = ref_max_scores(ops_all["f32"][0][:32], ops_all["f32"][1])
    bud = Budget(P["budget_s"])
    for qi in qs:
        if bud.expired():
            print(f"batch sweep stopped before Q={qi}: budget used")
            break
        sl = {fam: (qop[:qi], mop) for fam, (qop, mop) in ops_all.items()}
        for name in BATCH_KERNELS:
            if kernel_allowed(name, nb):
                run_kernel("scaling_Q", name, sl, q_all[:qi], mem_i8, rmax32, tg_all[:qi], nb, min(qi, 32), 0.25)
    show("scaling_Q", ["method", "n", "query_batch", "lat_p50_ms", "pipe_p50_ms", "qps", "us_per_query", "gflops_effective"], sort=["method", "query_batch"], max_rows=40)
    del ops_all
    gc.collect()

    # ---------------- single-request latency distribution (Q = 1)
    nl = min(ns[-1], 2 ** 14)
    mem_i8 = big[:nl]
    _, q1 = make_queries(mem_i8, 1, 0.25, rng_for(4, 3))
    ops1 = operands(mem_i8, q1)
    for name in ("f32_highest", "bf16_acc32", "packed_popcount"):
        if not kernel_allowed(name, nl):
            continue
        fn, fam, eb = KERNELS[name]
        qop, mop = ops1[fam]
        keep = []
        st = bench(fn, qop, mop, batch=1, samples=P["lat_samples"], modes=("latency",), keep=keep)
        LAT_SAMPLES[(name, "device_resident")] = np.array(keep)
        record("latency_dist", name, n=nl, d=D, query_batch=1, path="device_resident", **st)
        keep = []
        st = bench(lambda: np.asarray(fn(HOST_OPERAND[fam](q1.copy()), mop)), batch=1, samples=P["lat_samples"], modes=("latency",), keep=keep)
        LAT_SAMPLES[(name, "host_roundtrip")] = np.array(keep)
        record("latency_dist", name, n=nl, d=D, query_batch=1, path="host_roundtrip", **st)
    show("latency_dist", ["method", "path", "n", "n_samples", "first_call_ms", "lat_min_ms", "lat_p50_ms", "lat_p90_ms", "lat_p95_ms", "lat_p99_ms", "lat_p999_ms", "lat_max_ms"],
         sort=["method", "path"])
    if P["lat_samples"] < 100:
        print("Fewer than 100 latency samples in this profile: p99 / p99.9 are intentionally reported as NaN.")

    # ---------------- plots
    fig, axes = plt.subplots(1, 4, figsize=(19, 3.9))
    for name in SWEEP_KERNELS:
        g = sc[sc.method == name].sort_values("n")
        if len(g):
            axes[0].plot(g.n, g.pipe_p50_ms, "o-", label=name); axes[1].plot(g.n, g.gbps_memory, "o-", label=name)
    axes[0].axhline(DISPATCH_FLOOR_MS, color="gray", ls=":", label="dispatch floor")
    axes[0].set(xscale="log", yscale="log", xlabel="stored patterns N", ylabel="pipelined ms / call", title="time vs N"); axes[0].legend(fontsize=7)
    axes[1].set(xscale="log", yscale="log", xlabel="stored patterns N", ylabel="GB/s of memory read", title="achieved bandwidth")
    sq = results_df("scaling_Q")
    for name in BATCH_KERNELS:
        g = sq[sq.method == name].sort_values("query_batch")
        if len(g):
            axes[2].plot(g.query_batch, g.qps, "o-", label=name)
    axes[2].set(xscale="log", yscale="log", xlabel="batch size Q", ylabel="queries / s", title=f"throughput vs batch (N={nb})"); axes[2].legend(fontsize=7)
    for (name, path), arr in LAT_SAMPLES.items():
        if name == "f32_highest":
            axes[3].hist(arr * 1e3, bins=60, alpha=.55, label=path, log=True)
    axes[3].set(xscale="log", xlabel="latency (us)", ylabel="count", title=f"single-request latency (f32, N={nl})"); axes[3].legend(fontsize=7)
    for a in axes:
        a.grid(alpha=.25, which="both")
    plt.tight_layout(); savefig(fig, "s4_scaling"); plt.show()
else:
    print("Scaling study disabled.")

## 5. FlyHash-style sparse projection vs SimHash vs exact search

A sparse random projection maps a ±1 input to a larger space; winner-take-all (WTA) keeps the `k` largest of `m`
activations; similarity is the overlap of the binary codes. **This is a FlyHash-style proxy, not a recreation of the
report's `FlyHashCoder`.**

What v1 left open and this cell measures:

* **A control.** SimHash (sign of a dense Gaussian projection, compared by Hamming agreement) at the same bit budget
  shows how much of the FlyHash behaviour is the sparse-WTA design and how much is simply "a short hash".
* **Code-size accounting.** v1 reported dense float32 codes and an ideal bit-packed size. A k-of-m code has three
  honest sizes: dense bitmap (`m` bits), index list (`k·⌈log₂ m⌉` bits) and the information-theoretic bound
  `log₂ C(m,k)` bits. The raw ±1 input needs only `D` bits, so the table shows when the "expanded" code is *larger*
  than the data it summarises.
* **Ties.** Activations of a ±1 input through ±1/√fan-in weights take few distinct values, so many units tie at the
  WTA cut-off and `top_k` breaks the ties by unit index. `boundary_tie_fraction` reports how often this happens.
  (Replacing the weights with Gaussians removes most ties; the sweep shows whether recall changes.)
* **Variants:** larger expansion `m`, larger `k`, larger fan-in.

In [ ]:
#@title 5. FlyHash vs SimHash vs exact
clear("fly_vs_simhash"); clear("fly_timing")


def fly_projection(rng, d, m, fanin, kind="binary"):
    rows = np.argpartition(rng.random((m, d)), fanin - 1, axis=1)[:, :fanin]               # distinct inputs per unit
    vals = (rng.choice([-1.0, 1.0], size=(m, fanin)) if kind == "binary" else rng.normal(size=(m, fanin))) / math.sqrt(fanin)
    out = np.zeros((d, m), np.float32)
    out[rows, np.arange(m)[:, None]] = vals
    return out


@partial(jax.jit, static_argnames=("k",))
def wta_encode(x, proj, k):
    a = jnp.matmul(x, proj, precision=HI)
    idx = lax.top_k(a, k)[1]
    return jnp.zeros(a.shape, jnp.float32).at[jnp.arange(a.shape[0])[:, None], idx].set(1.0)


@partial(jax.jit, static_argnames=("k",))
def wta_boundary_ties(x, proj, k):
    top = lax.top_k(jnp.matmul(x, proj, precision=HI), k + 1)[0]
    return jnp.mean((top[:, k - 1] == top[:, k]).astype(jnp.float32))


@jax.jit
def sign_encode(x, proj):
    return jnp.where(jnp.matmul(x, proj, precision=HI) >= 0, 1.0, -1.0)


@jax.jit
def code_search(qc, mc):
    s = jnp.matmul(qc, mc.T, precision=HI)
    return jnp.argmax(s, axis=1), lax.top_k(s, 10)[1]


FLY_METHODS = {
    "exact_flat": dict(kind="exact"),
    "simhash_512b": dict(kind="sim", bits=512),
    "simhash_2048b": dict(kind="sim", bits=2048),
    "fly_m2048_k64_f16 (v1)": dict(kind="fly", m=2048, k=64, fanin=16, w="binary"),
    "fly_gauss_m2048_k64_f16": dict(kind="fly", m=2048, k=64, fanin=16, w="gauss"),
    "fly_m2048_k128_f16": dict(kind="fly", m=2048, k=128, fanin=16, w="binary"),
    "fly_m8192_k256_f16": dict(kind="fly", m=8192, k=256, fanin=16, w="binary"),
    "fly_m2048_k64_f64": dict(kind="fly", m=2048, k=64, fanin=64, w="binary"),
}
if PROFILE == "smoke":
    FLY_METHODS = {k: v for k, v in FLY_METHODS.items() if k in ("exact_flat", "simhash_512b", "fly_m2048_k64_f16 (v1)", "fly_gauss_m2048_k64_f16")}


def code_sizes(spec):
    if spec["kind"] == "exact":
        return dict(code_bits=D, active_units=D, bitmap_bytes=D / 8, index_list_bytes=D / 8, ideal_bytes=D / 8)
    if spec["kind"] == "sim":
        return dict(code_bits=spec["bits"], active_units=spec["bits"], bitmap_bytes=spec["bits"] / 8, index_list_bytes=spec["bits"] / 8, ideal_bytes=spec["bits"] / 8)
    m, k = spec["m"], spec["k"]
    return dict(code_bits=m, active_units=k, bitmap_bytes=m / 8, index_list_bytes=k * math.ceil(math.log2(m)) / 8, ideal_bytes=ideal_sparse_code_bytes(m, k))


def build_encoder(spec, rng):
    if spec["kind"] == "exact":
        return lambda x: x
    if spec["kind"] == "sim":
        R = put(rng.normal(size=(D, spec["bits"])).astype(np.float32))
        return lambda x: sign_encode(x, R)
    pr = put(fly_projection(rng, D, spec["m"], spec["fanin"], spec["w"]))
    return lambda x: wta_encode(x, pr, spec["k"]), pr


if RUN_FLY:
    fly_ns = [n for n in N_VALUES if n >= 256][:2] or N_VALUES[:1]
    acc, ties = {}, {}
    for n in fly_ns:
        for si in SEEDS:
            mem_np = bipolar_i8(n, D, rng_for(5, 0, n, si)); mem32 = to_f32(put(mem_np))
            qsets = {}
            for noise in P["fly_noise"]:
                tg, q_np = make_queries(mem_np, P["Q"], noise, rng_for(5, 1, n, int(round(noise * 1000)), si))
                qsets[noise] = (tg, to_f32(put(q_np)))
            for mi, (name, spec) in enumerate(FLY_METHODS.items()):
                enc = build_encoder(spec, rng_for(5, 2, mi, si))
                pr = None
                if isinstance(enc, tuple):
                    enc, pr = enc
                mc = enc(mem32)
                for noise, (tg, q32) in qsets.items():
                    i1, i10 = (np.asarray(a) for a in code_search(enc(q32), mc))
                    d = acc.setdefault((n, name, noise), {"c1": [], "c10": []})
                    d["c1"].append(i1 == tg); d["c10"].append((i10 == tg[:, None]).any(1))
                    if pr is not None and noise == 0.25:
                        ties.setdefault((n, name), []).append(float(wta_boundary_ties(q32, pr, spec["k"])))
    for (n, name, noise), d in sorted(acc.items()):
        r1, lo, hi, trials = pooled_recall(d["c1"]); r10, lo10, hi10, _ = pooled_recall(d["c10"])
        extra = dict(analytic_recall=analytic_flat_recall(D, n, noise)) if name == "exact_flat" else {}
        spec = FLY_METHODS[name]
        record("fly_vs_simhash", name, n=n, d=D, noise=noise, query_batch=P["Q"], trials=trials, recall_at_1=r1, recall_at_1_lo=lo, recall_at_1_hi=hi,
               recall_at_10=r10, recall_at_10_lo=lo10, recall_at_10_hi=hi10, boundary_tie_fraction=float(np.mean(ties[(n, name)])) if (n, name) in ties else NAN,
               **code_sizes(spec), **extra)
    fv = results_df("fly_vs_simhash")
    n0 = fly_ns[0]
    noise_ref = min(P["fly_noise"], key=lambda x: abs(x - 0.25))
    pivot = fv[fv.n == n0].pivot_table(index="method", columns="noise", values="recall_at_1").round(3)
    print(f"recall@1 at N={n0} (rows: method, columns: fraction of query bits flipped)")
    display(pivot)
    show("fly_vs_simhash", ["method", "code_bits", "active_units", "bitmap_bytes", "index_list_bytes", "ideal_bytes", "boundary_tie_fraction"], sort=["method"], n=n0, noise=noise_ref)
    print(f"raw +-1 input: {D / 8:.0f} bytes/item bit-packed. A code whose 'ideal_bytes' exceeds that is larger than the data it encodes.")
    base = fv[(fv.method == "exact_flat") & (fv.n == n0)]
    zz = [abs(r.recall_at_1 - r.analytic_recall) <= 4.5 * math.sqrt(max(r.analytic_recall * (1 - r.analytic_recall), 1e-12) / r.trials) + 1.0 / r.trials for r in base.itertuples()]
    check("S5 exact-search recall in the FlyHash comparison matches the analytic model", all(zz), f"{len(zz)} noise levels at N={n0}")
    ones = np.asarray(wta_encode(to_f32(put(bipolar_i8(32, D, rng_for(5, 9)))), put(fly_projection(rng_for(5, 8), D, 2048, 16)), 64)).sum(1)
    check("S5 every WTA code has exactly k active units", bool((ones == 64).all()), f"counts {sorted(set(ones.astype(int).tolist()))}")

    # timing of encode + search at the largest N of this section (device resident)
    n_t = fly_ns[-1]
    mem_np = bipolar_i8(n_t, D, rng_for(5, 3)); _, q_np = make_queries(mem_np, P["Q"], 0.25, rng_for(5, 4))
    m32, q32 = to_f32(put(mem_np)), to_f32(put(q_np))
    for mi, (name, spec) in enumerate(FLY_METHODS.items()):
        enc = build_encoder(spec, rng_for(5, 2, mi, 0)); enc = enc[0] if isinstance(enc, tuple) else enc
        mc = enc(m32)
        fn = (lambda q, enc=enc, mc=mc: code_search(enc(q), mc))
        record("fly_timing", name, n=n_t, d=D, query_batch=P["Q"], **bench(fn, q32, batch=P["Q"]), **code_sizes(spec))
    show("fly_timing", ["method", "n", "pipe_p50_ms", "qps", "code_bits", "bitmap_bytes"], sort=["pipe_p50_ms"])

    fig, axes = plt.subplots(1, len(fly_ns), figsize=(6.2 * len(fly_ns), 4), sharey=True, squeeze=False)
    for ax, n in zip(axes[0], fly_ns):
        for name in FLY_METHODS:
            g = fv[(fv.n == n) & (fv.method == name)].sort_values("noise")
            ax.errorbar(g.noise, g.recall_at_1, yerr=yerr(g.recall_at_1, g.recall_at_1_lo, g.recall_at_1_hi), marker="o", ms=3, capsize=1.5, label=name, lw=1.2)
        ax.set(title=f"N={n}", xlabel="fraction of query bits flipped"); ax.grid(alpha=.25)
    axes[0][0].set_ylabel("recall@1"); axes[0][-1].legend(fontsize=6.5)
    plt.tight_layout(); savefig(fig, "s5_flyhash"); plt.show()
else:
    print("FlyHash study disabled.")

## 6. Role–filler VSA

**A. Structured record retrieval.** Five-field records (department, role, location, team, unique ID), each field bound to its
role vector and bundled into one ±1 record. The cue is the bundle of the first three fields. v1 reported ID top-1 and
compared it with an "oracle"; but all records sharing a cue tuple have *identical* cue vectors, so any deterministic
system returns the same record for all of them and its top-1 equals `distinct tuples / records` by construction. The
informative quantities are therefore **tuple-level accuracy** (does the retrieved record share the cue's three fields?)
and how it degrades with dimension.

**B. Bundle-decode capacity.** Unbind one role from a bundle of `k` role–filler pairs and clean up against a codebook of `F`
fillers. A majority bundle of `k` terms correlates with each term by `C(k-1,(k-1)/2)/2^(k-1)`, which makes decoding
equivalent to flat retrieval with an effective flip probability `(1-c_k)/2`; the closed form from Section A of the helpers
therefore predicts decode accuracy for every `(D, k)`, and the measurements must agree with it.

In [ ]:
#@title 6. VSA: record retrieval and bundle-decode capacity
clear("vsa_records"); clear("vsa_decode")


@jax.jit
def vsa_top1(records, cues):
    s = jnp.matmul(cues, records.T, precision=HI)
    return jnp.argmax(s, axis=1), lax.top_k(s, 10)[1]


def vsa_make_records(rng, d, n_rec):
    sizes = [6, 8, 6, 10]
    fields = np.stack([rng.integers(0, s, size=n_rec) for s in sizes] + [np.arange(n_rec)], axis=1)
    roles = bipolar_i8(5, d, rng)
    fillers = [bipolar_i8(s, d, rng) for s in sizes + [n_rec]]
    bound = [roles[j][None, :].astype(np.int16) * fillers[j][fields[:, j]].astype(np.int16) for j in range(5)]
    rec = np.where(sum(bound) >= 0, 1, -1).astype(np.float32)           # five terms: never zero
    cue = np.where(sum(bound[:3]) >= 0, 1, -1).astype(np.float32)       # three terms: never zero
    return fields, rec, cue


@partial(jax.jit, static_argnames=("k",))
def vsa_bundle(roles, codebook, idx, k):
    acc = jnp.zeros((idx.shape[0], roles.shape[1]), jnp.int32)
    for j in range(k):
        acc = acc + roles[j][None, :].astype(jnp.int32) * codebook[idx[:, j]].astype(jnp.int32)
    return jnp.where(acc >= 0, 1.0, -1.0)


@jax.jit
def vsa_decode_first(bundle, role0, codebook):
    u = bundle * role0[None, :].astype(jnp.float32)
    return jnp.argmax(jnp.matmul(u, codebook.astype(jnp.float32).T, precision=HI), axis=1)


if RUN_VSA:
    # ---------------- A. record retrieval
    for dd in P["vsa_dims"]:
        accA = {k: [] for k in ("tuple", "id1", "id10", "uniq")}
        distinct, ambiguous = [], []
        for si in SEEDS:
            fields, rec, cue = vsa_make_records(rng_for(6, 0, dd, si), dd, P["vsa_records"])
            pred, top10 = (np.asarray(a) for a in vsa_top1(put(rec), put(cue)))
            truth = np.arange(len(rec))
            tuples = [tuple(r) for r in fields[:, :3]]
            cnt = Counter(tuples); uniq = np.array([cnt[t] == 1 for t in tuples])
            accA["tuple"].append(np.array([tuples[int(p)] == t for p, t in zip(pred, tuples)]))
            accA["id1"].append(pred == truth); accA["id10"].append((top10 == truth[:, None]).any(1)); accA["uniq"].append((pred == truth)[uniq])
            distinct.append(len(cnt) / len(rec)); ambiguous.append(1 - uniq.mean())
        t1, tlo, thi, trials = pooled_recall(accA["tuple"])
        record("vsa_records", "vsa_proxy", n=P["vsa_records"], d=dd, trials=trials, tuple_level_accuracy=t1, tuple_level_lo=tlo, tuple_level_hi=thi,
               id_top1=float(np.concatenate(accA["id1"]).mean()), id_top10=float(np.concatenate(accA["id10"]).mean()),
               unique_cue_id_top1=float(np.concatenate(accA["uniq"]).mean()) if sum(len(a) for a in accA["uniq"]) else NAN,
               oracle_id_top1_distinct_over_n=float(np.mean(distinct)), ambiguous_cue_fraction=float(np.mean(ambiguous)))
    show("vsa_records", ["d", "n", "trials", "tuple_level_accuracy", "tuple_level_lo", "tuple_level_hi", "id_top1", "oracle_id_top1_distinct_over_n", "id_top10",
                         "unique_cue_id_top1", "ambiguous_cue_fraction"], sort=["d"])

    # ---------------- B. decode capacity vs bundle size and dimension
    F = 64
    for dd in P["vsa_dims"]:
        for k in P["vsa_ks"]:
            correct = []
            for si in SEEDS:
                r = rng_for(6, 1, dd, k, si)
                roles = bipolar_i8(k, dd, r); codebook = bipolar_i8(F, dd, r)
                idx = r.integers(0, F, size=(P["vsa_trials"], k)).astype(np.int32)
                bundle = vsa_bundle(put(roles), put(codebook), put(idx), k)
                correct.append(np.asarray(vsa_decode_first(bundle, put(roles[0]), put(codebook))) == idx[:, 0])
            r1, lo, hi, trials = pooled_recall(correct)
            ana = analytic_vsa_decode(dd, F, k)
            record("vsa_decode", "role_unbind_cleanup", n=F, d=dd, scenario=f"k={k}", trials=trials, bundle_terms=k, recall_at_1=r1, recall_at_1_lo=lo, recall_at_1_hi=hi,
                   analytic_recall=ana, z_vs_analytic=(r1 - ana) / math.sqrt(max(ana * (1 - ana), 1e-12) / trials), bundle_corr=vsa_bundle_corr(k))
    vd = results_df("vsa_decode")
    odd = vd[(vd.analytic_recall > 1e-3) & (vd.analytic_recall < 1 - 1e-3)]
    viol = vd[abs(vd.recall_at_1 - vd.analytic_recall) > 4.5 * np.sqrt(np.maximum(vd.analytic_recall * (1 - vd.analytic_recall), 1e-12) / vd.trials) + 1.0 / vd.trials]
    check("S6 VSA decode accuracy matches the closed-form prediction for every (D, k)", len(viol) == 0, f"{len(vd)} cells, {len(odd)} informative; violations: {len(viol)}")
    print(vd.pivot_table(index="bundle_terms", columns="d", values="recall_at_1").round(3).to_string())
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.9))
    for dd in P["vsa_dims"]:
        g = vd[vd.d == dd].sort_values("bundle_terms")
        axes[0].errorbar(g.bundle_terms, g.recall_at_1, yerr=yerr(g.recall_at_1, g.recall_at_1_lo, g.recall_at_1_hi), marker="o", capsize=2, label=f"D={dd}")
        axes[0].plot(g.bundle_terms, g.analytic_recall, "k:", lw=1)
    axes[0].set(xlabel="role-filler pairs bundled (k)", ylabel="decode accuracy (F=64 fillers)", title="measured (points) vs closed form (dotted)"); axes[0].legend(); axes[0].grid(alpha=.25)
    ra = results_df("vsa_records").sort_values("d")
    axes[1].errorbar(ra.d, ra.tuple_level_accuracy, yerr=yerr(ra.tuple_level_accuracy, ra.tuple_level_lo, ra.tuple_level_hi), marker="o", capsize=2, label="tuple-level accuracy")
    axes[1].plot(ra.d, ra.id_top1, "s--", label="ID top-1"); axes[1].plot(ra.d, ra.oracle_id_top1_distinct_over_n, "k:", label="ID top-1 bound (distinct/N)")
    axes[1].set(xscale="log", xlabel="dimension D", ylabel="accuracy", title="record retrieval from a 3-field cue"); axes[1].legend(fontsize=8); axes[1].grid(alpha=.25)
    plt.tight_layout(); savefig(fig, "s6_vsa"); plt.show()
else:
    print("VSA study disabled.")

## 7. Sparse distributed memory (SDM)

`M` random ±1 hard-location addresses; a pattern activates its `K` nearest locations (fixed-K variant, **not** the
report's exact address-activation rule); writes add the value to the activated counters; a read sums the activated
counters and takes the sign. Autoassociative here: key = value.

What this section adds to v1:

* **Two implementations**, verified bit-for-bit equal: v1's dense `mask.T @ values` (cost O(n·M·D), materialises an
  n×M mask) and a scatter-add write / gather read (cost O(n·K·D)). Which is faster depends on the hardware.
* **Capacity sweeps** over `M`, `K`, load and noise, with Wilson intervals and optional iterated reads.
* **Output BER vs input noise.** In v1 at 15 % input noise the SDM's output had *more* bit errors than its input for n ≥ 100;
  `improves_over_input` makes that visible.
* **Memory efficiency.** Bytes per *reliably stored* pattern (90 % exact recall), against a flat bit-packed memory,
  whose retrieval success at the same noise is given by the analytic column.

In [ ]:
#@title 7. SDM: implementations, capacity, noise tolerance, bytes per pattern
clear("sdm_impl"); clear("sdm_capacity"); clear("sdm_summary")


@partial(jax.jit, static_argnames=("k",))
def sdm_locations(keys, addresses, k):
    return lax.top_k(jnp.matmul(keys, addresses.T, precision=HI), k)[1]


@partial(jax.jit, static_argnames=("k",))
def sdm_write_dense(keys, values, addresses, k):          # v1's formulation
    loc = sdm_locations(keys, addresses, k)
    mask = jnp.zeros((keys.shape[0], addresses.shape[0]), jnp.float32).at[jnp.arange(keys.shape[0])[:, None], loc].set(1.0)
    return jnp.matmul(mask.T, values, precision=HI)


@partial(jax.jit, static_argnames=("k",))
def sdm_write_scatter(keys, values, addresses, k):
    loc = sdm_locations(keys, addresses, k)
    return jnp.zeros((addresses.shape[0], values.shape[1]), jnp.float32).at[loc.reshape(-1)].add(jnp.repeat(values, k, axis=0))


@partial(jax.jit, static_argnames=("k",))
def sdm_read_dense(queries, addresses, counters, k):
    loc = sdm_locations(queries, addresses, k)
    mask = jnp.zeros((queries.shape[0], addresses.shape[0]), jnp.float32).at[jnp.arange(queries.shape[0])[:, None], loc].set(1.0)
    return jnp.where(jnp.matmul(mask, counters, precision=HI) >= 0, 1.0, -1.0)


@partial(jax.jit, static_argnames=("k",))
def sdm_read_gather(queries, addresses, counters, k):
    loc = sdm_locations(queries, addresses, k)
    return jnp.where(counters[loc].sum(axis=1) >= 0, 1.0, -1.0)


def sdm_read_iter(x, addresses, counters, k, iters):
    for _ in range(iters):
        x = sdm_read_gather(x, addresses, counters, k)
    return x


if RUN_SDM:
    Ds = P["sdm_dim"]
    # ---------------- (a) dense mask vs scatter/gather: equality and speed
    n_impl = P["sdm_loads"][len(P["sdm_loads"]) // 2]
    k_impl = P["sdm_K"][len(P["sdm_K"]) // 2]
    all_equal = True
    for M in sorted({P["sdm_M"][0], P["sdm_M"][-1]}):
        addr = to_f32(put(bipolar_i8(M, Ds, rng_for(7, 0, M)))); pat = to_f32(put(bipolar_i8(n_impl, Ds, rng_for(7, 1, M))))
        c_d, c_s = sdm_write_dense(pat, pat, addr, k_impl), sdm_write_scatter(pat, pat, addr, k_impl)
        r_d, r_s = sdm_read_dense(pat, addr, c_d, k_impl), sdm_read_gather(pat, addr, c_s, k_impl)
        eq = bool(jnp.array_equal(c_d, c_s)) and bool(jnp.array_equal(r_d, r_s))
        all_equal &= eq
        for name, (wf, wargs, rf, rargs, ctr) in {"dense_mask": (sdm_write_dense, (pat, pat, addr, k_impl), sdm_read_dense, (pat, addr, c_d, k_impl), c_d),
                                                  "scatter_gather": (sdm_write_scatter, (pat, pat, addr, k_impl), sdm_read_gather, (pat, addr, c_s, k_impl), c_s)}.items():
            w = bench(wf, *wargs, batch=n_impl); rd = bench(rf, *rargs, batch=n_impl)
            record("sdm_impl", name, n=n_impl, d=Ds, variant=M, query_batch=n_impl, hard_locations=M, k_active=k_impl, identical_to_dense=eq,
                   write_pipe_ms=w["pipe_p50_ms"], write_lat_ms=w["lat_p50_ms"], read_pipe_ms=rd["pipe_p50_ms"], read_lat_ms=rd["lat_p50_ms"],
                   write_first_call_ms=w["first_call_ms"], temp_mask_bytes=n_impl * M * 4 if name == "dense_mask" else 0, counters_bytes=nbytes(ctr))
    check("S7 scatter/gather SDM is bit-identical to the dense-mask SDM", all_equal)
    show("sdm_impl", ["method", "hard_locations", "n", "k_active", "write_pipe_ms", "read_pipe_ms", "write_first_call_ms", "temp_mask_bytes", "identical_to_dense"], sort=["hard_locations", "method"])

    # ---------------- (b) capacity / noise sweep
    acc, maxabs = {}, {}
    bud = Budget(P["budget_s"])
    for M in P["sdm_M"]:
        for si in SEEDS:
            addr = to_f32(put(bipolar_i8(M, Ds, rng_for(7, 2, M, si))))
            for n in P["sdm_loads"]:
                if bud.expired():
                    break
                pats = bipolar_i8(n, Ds, rng_for(7, 3, M, n, si)); p32 = to_f32(put(pats))
                for K in P["sdm_K"]:
                    ctr = sdm_write_scatter(p32, p32, addr, K)
                    maxabs[(M, K, n)] = max(maxabs.get((M, K, n), 0), float(jnp.max(jnp.abs(ctr))))
                    for noise in P["sdm_noise"]:
                        q = to_f32(put(flip_i8(pats, noise, rng_for(7, 4, M, K, n, int(noise * 1000), si))))
                        for iters in ((1,) if noise == 0 else (1, 3)):
                            out = np.asarray(sdm_read_iter(q, addr, ctr, K, iters))
                            d = acc.setdefault((M, K, n, noise, iters), {"exact": [], "ber": []})
                            d["exact"].append((out == pats).all(axis=1)); d["ber"].append((out != pats).mean(axis=1))
    if bud.expired():
        print(f"SDM capacity sweep cut short after {bud.s:.0f}s: later (M, load) cells are missing")
    for (M, K, n, noise, iters), d in sorted(acc.items()):
        r1, lo, hi, trials = pooled_recall(d["exact"])
        ber = float(np.concatenate(d["ber"]).mean())
        record("sdm_capacity", "sdm_fixed_k", n=n, d=Ds, noise=noise, variant=f"M{M}_K{K}_it{iters}", hard_locations=M, k_active=K, read_iters=iters, trials=trials,
               exact_pattern_recall=r1, exact_pattern_recall_lo=lo, exact_pattern_recall_hi=hi, bit_error_rate_out=ber, ber_in=noise,
               improves_over_input=bool(ber < noise) if noise > 0 else NAN, max_abs_counter=maxabs.get((M, K, n), NAN),
               flat_search_recall_analytic=analytic_flat_recall(Ds, n, noise), load_per_location=n * K / M)
    cap = results_df("sdm_capacity")
    for (M, K), g in cap[(cap.noise == 0) & (cap.read_iters == 1)].groupby(["hard_locations", "k_active"]):
        ok = g[g.exact_pattern_recall >= 0.9]
        n90 = int(ok.n.max()) if len(ok) else 0
        mx = float(g[g.n <= max(n90, 1)].max_abs_counter.max()) if n90 else NAN
        counter_bytes = M * Ds * (1 if mx < 127 else 2 if mx < 32767 else 4)
        sdm_bytes = counter_bytes + M * Ds / 8                                # counters + bit-packed addresses
        record("sdm_summary", "sdm_fixed_k", d=Ds, variant=f"M{M}_K{K}", hard_locations=M, k_active=K, reliable_patterns_n90=n90, max_abs_counter_at_n90=mx,
               sdm_bytes_int_counters=sdm_bytes, sdm_bytes_f32=2 * M * Ds * 4, bytes_per_reliable_pattern=(sdm_bytes / n90) if n90 else NAN,
               flat_bitpacked_bytes_per_pattern=Ds / 8, flat_f32_bytes_per_pattern=Ds * 4, memory_overhead_vs_flat_bitpacked=(sdm_bytes / n90) / (Ds / 8) if n90 else NAN)
    show("sdm_summary", ["hard_locations", "k_active", "reliable_patterns_n90", "max_abs_counter_at_n90", "sdm_bytes_int_counters", "bytes_per_reliable_pattern",
                         "flat_bitpacked_bytes_per_pattern", "memory_overhead_vs_flat_bitpacked"], sort=["hard_locations", "k_active"])
    noisy = cap[(cap.noise > 0) & (cap.read_iters == 1)]
    worse = noisy[noisy.improves_over_input == False]  # noqa: E712
    print(f"{len(worse)}/{len(noisy)} noisy-cue cells return a pattern with MORE bit errors than the cue it was given (single read).")
    show("sdm_capacity", ["variant", "n", "noise", "read_iters", "load_per_location", "exact_pattern_recall", "bit_error_rate_out", "ber_in", "improves_over_input",
                          "flat_search_recall_analytic"], sort=["variant", "noise", "read_iters", "n"], max_rows=40)

    fig, axes = plt.subplots(1, 3, figsize=(16, 3.9))
    for (M, K), g in cap[(cap.noise == 0) & (cap.read_iters == 1)].groupby(["hard_locations", "k_active"]):
        g = g.sort_values("n")
        axes[0].errorbar(g.n, g.exact_pattern_recall, yerr=yerr(g.exact_pattern_recall, g.exact_pattern_recall_lo, g.exact_pattern_recall_hi), marker="o", ms=3, capsize=1.5, label=f"M={M} K={K}")
    fl = cap[(cap.noise == 0)].sort_values("n").drop_duplicates("n")
    axes[0].plot(fl.n, fl.flat_search_recall_analytic, "k--", lw=1, label="flat search (analytic)")
    axes[0].set(xscale="log", xlabel="stored patterns", ylabel="exact whole-pattern recall", title="SDM capacity (noise-free cue)"); axes[0].legend(fontsize=6); axes[0].grid(alpha=.25)
    nz = [x for x in P["sdm_noise"] if x > 0][-1]
    for (M, K), g in cap[(cap.noise == nz) & (cap.read_iters == 1)].groupby(["hard_locations", "k_active"]):
        g = g.sort_values("n"); axes[1].plot(g.n, g.bit_error_rate_out, "o-", ms=3, label=f"M={M} K={K}")
    axes[1].axhline(nz, color="k", ls="--", lw=1, label="input BER"); axes[1].set(xscale="log", xlabel="stored patterns", ylabel="output bit-error rate", title=f"denoising at {int(nz * 100)}% input noise (single read)")
    axes[1].legend(fontsize=6); axes[1].grid(alpha=.25)
    im = results_df("sdm_impl")
    for name in ("dense_mask", "scatter_gather"):
        g = im[im.method == name].sort_values("hard_locations")
        axes[2].plot(g.hard_locations, g.read_pipe_ms, "o-", label=f"read {name}"); axes[2].plot(g.hard_locations, g.write_pipe_ms, "s--", label=f"write {name}")
    axes[2].set(xscale="log", yscale="log", xlabel="hard locations M", ylabel="pipelined ms", title="implementation cost"); axes[2].legend(fontsize=7); axes[2].grid(alpha=.25, which="both")
    plt.tight_layout(); savefig(fig, "s7_sdm"); plt.show()
else:
    print("SDM study disabled.")

## 8. Out-of-distribution rejection

In v1 the OOD queries were independent random vectors; they score about 0.12 maximum cosine against the memory while a
25 %-noise known query scores about 0.5, a gap of roughly nine standard deviations, so AUROC = 1.0 for *any* sensible
score and the test could not fail. v2 uses families that are progressively harder, and four scores:

| family | construction | expected behaviour |
|---|---|---|
| `random` | independent ±1 vectors | trivial |
| `chimera` | half the bits from stored pattern A, half from B | max-cosine ≈ 0.5, same as a 25 %-noise known query, so `max_cos` should fail and the *margin* should work |
| `partial_f` | fraction `f` copied from one stored pattern, rest random | cosine ≈ f; at `f = 0.5` it is statistically identical to a 25 %-noise known query, so AUROC must be 0.5 (a built-in sanity check of the metric) |

Scores (higher = more "known"): `max_cos`, `margin_top1_top2`, `neg_entropy` of the Hopfield attention weights, `logsumexp`
(negative Hopfield energy). Thresholds are calibrated on a *separate* set of known queries (5th percentile → nominal 95 % TPR)
and evaluated on independent known and OOD queries. AUROC intervals are bootstrap intervals.

In [ ]:
#@title 8. OOD rejection: families x scores
clear("ood")


@jax.jit
def ood_scores(mem, q, beta):
    s = jnp.matmul(q, mem.T, precision=HI) / mem.shape[1]
    t2 = lax.top_k(s, 2)[0]
    w = jax.nn.softmax(beta * s, axis=-1)
    return jnp.stack([t2[:, 0], t2[:, 0] - t2[:, 1], jnp.sum(w * jnp.log(w + 1e-30), axis=-1), jax.nn.logsumexp(beta * s, axis=-1) / beta], axis=1)


SCORE_NAMES = ["max_cos", "margin_top1_top2", "neg_entropy", "logsumexp"]
PARTIALS = [0.1, 0.2, 0.3, 0.4, 0.5]
OOD_FAMILIES = ["random", "chimera"] + [f"partial_{f}" for f in PARTIALS]


def ood_queries(name, mem_i8, n_q, rng):
    n, d = mem_i8.shape
    if name.startswith("known_p"):
        return flip_i8(mem_i8[rng.integers(0, n, n_q)], int(name[len("known_p"):]) / 100.0, rng)
    if name == "random":
        return bipolar_i8(n_q, d, rng)
    if name == "chimera":
        a = rng.integers(0, n, n_q)
        b = (a + 1 + rng.integers(0, n - 1, n_q)) % n
        return np.where(rng.random((n_q, d)) < 0.5, mem_i8[a], mem_i8[b]).astype(np.int8)
    f = float(name.split("_")[1])
    return np.where(rng.random((n_q, d)) < f, mem_i8[rng.integers(0, n, n_q)], bipolar_i8(n_q, d, rng)).astype(np.int8)


def auroc_ci(pos, neg, n_boot=150, seed=0):
    rng = np.random.default_rng(seed)
    vals = [auroc(pos[rng.integers(0, len(pos), len(pos))], neg[rng.integers(0, len(neg), len(neg))]) for _ in range(n_boot)]
    return tuple(np.quantile(vals, [0.025, 0.975]))


if RUN_OOD:
    n_o, q_o = P["ood_n"], P["ood_q"]
    sets = {k: [] for k in ["known_p25_cal", "known_p25", "known_p40"] + OOD_FAMILIES}
    for si in SEEDS:
        mem_i8 = bipolar_i8(n_o, D, rng_for(8, 0, si)); mem = to_f32(put(mem_i8))
        for fam in sets:
            base = "known_p25" if fam.startswith("known_p25") else fam
            q = ood_queries(base, mem_i8, q_o, rng_for(8, 1, si, list(sets).index(fam)))
            sets[fam].append(np.asarray(ood_scores(mem, to_f32(put(q)), BETA)))
    S = {k: np.concatenate(v) for k, v in sets.items()}
    rows = []
    for si_, sname in enumerate(SCORE_NAMES):
        thr = float(np.quantile(S["known_p25_cal"][:, si_], 0.05))
        tpr = float(np.mean(S["known_p25"][:, si_] >= thr))
        for fam in OOD_FAMILIES + ["known_p40"]:
            pos, neg = (S["known_p25"], S[fam]) if fam != "known_p40" else (S["known_p40"], S["random"])
            pos_s, neg_s = pos[:, si_], neg[:, si_]
            a = auroc(pos_s, neg_s)
            lo, hi = auroc_ci(pos_s, neg_s, seed=si_)
            rows.append(record("ood", sname, scenario=("known25_vs_" + fam) if fam != "known_p40" else "known40_vs_random", n=n_o, d=D, query_batch=len(pos_s), auroc=a, auroc_lo=lo, auroc_hi=hi,
                               calibrated_threshold=thr, tpr_known=tpr, tpr_pos=float(np.mean(pos_s >= thr)), fpr_ood=float(np.mean(neg_s >= thr)),
                               n_pos=len(pos_s), n_neg=len(neg_s)))
    od = results_df("ood")
    piv = od.pivot_table(index="scenario", columns="method", values="auroc")[SCORE_NAMES].round(3)
    print(f"AUROC (positives = known queries; {len(S['known_p25'])} per set pooled over {len(SEEDS)} seeds). 0.5 = chance.")
    display(piv)
    pf = od.pivot_table(index="scenario", columns="method", values="fpr_ood")[SCORE_NAMES].round(3)
    print("False-positive rate of the calibrated threshold (OOD queries wrongly accepted; target is small when TPR ~ 0.95):")
    display(pf)
    print("TPR of the calibrated thresholds on independent known queries:", {r.method: round(r.tpr_known, 3) for r in od.drop_duplicates('method').itertuples()})
    half = od[(od.scenario == "known25_vs_partial_0.5") & (od.method == "max_cos")].iloc[0]
    check("S8 partial_0.5 is indistinguishable from known-25%-noise (AUROC of max_cos within [0.45, 0.55] and CI contains 0.5)",
          0.45 <= half.auroc <= 0.55 and half.auroc_lo <= 0.5 <= half.auroc_hi, f"AUROC={half.auroc:.3f} CI=[{half.auroc_lo:.3f},{half.auroc_hi:.3f}]")
    rnd = od[(od.scenario == "known25_vs_random")]
    print(f"known25 vs random is {'trivial' if (rnd.auroc > 0.999).all() else 'not trivial'} here (AUROC {rnd.auroc.min():.4f}-{rnd.auroc.max():.4f}); harder families are the informative ones.")
    fig, ax = plt.subplots(figsize=(6.6, 3.9))
    im = ax.imshow(piv.values, vmin=0.4, vmax=1.0, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(SCORE_NAMES)), SCORE_NAMES, rotation=20); ax.set_yticks(range(len(piv.index)), piv.index, fontsize=7)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i, j]:.2f}", ha="center", va="center", color="w", fontsize=7)
    fig.colorbar(im, label="AUROC"); ax.set_title("OOD AUROC by family and score"); plt.tight_layout(); savefig(fig, "s8_ood"); plt.show()
else:
    print("OOD study disabled.")

## 9. Hardware fault-injection simulation

In v1 every fault condition (up to 5 % stored bit flips and Gaussian noise with σ = 0.5) left recall at 1.000, because
the operating point (N = 256, 20 % query noise) has an error probability near 1e-11: the test was saturated.

**v2 first chooses an operating point from the analytic model**: the query-noise level at which fault-free exact search has
95 % recall for this `N` and `D`. Degradation in either direction is then visible, and the fault-free row is checked against
the model. Faults are applied to the *stored* memory (static) or to the *read-out* (dynamic):

| mode | meaning | severity |
|---|---|---|
| `bitflip` | stored bits inverted | probability |
| `stuck_zero` | cell reads 0 (open) | fraction of cells |
| `stuck_pos` | cell stuck at +1 | fraction of cells |
| `dead_rows` | a whole stored pattern unreadable | fraction of patterns |
| `dead_cols` | a whole dimension unreadable for every pattern | fraction of dimensions |
| `gauss_static` | fabrication-style additive variation on each stored value | σ |
| `lognormal_gain` | multiplicative per-cell gain `exp(N(0, σ²))` | σ |
| `read_noise` | fresh Gaussian noise on every similarity score at every query | σ in cosine units |
| `adc_bits` | similarity scores quantised to 2^b levels over [-1, 1] | bits (lower is worse) |

These are **functional** fault models. They are not calibrated FeNAND, crossbar or compute-in-memory device physics, and
say nothing about energy, endurance, retention, line resistance or ADC/DAC power.

In [ ]:
#@title 9. Fault injection at an informative operating point
clear("fault_injection")


def _fscores(mem, q, noise, adc_step):
    s = jnp.matmul(q, mem.T, precision=HI) / mem.shape[1] + noise
    return jnp.where(adc_step > 0, jnp.round((s + 1.0) / jnp.maximum(adc_step, 1e-9)) * adc_step - 1.0, s)


@jax.jit
def faulty_exact(mem, q, noise, adc_step):
    return jnp.argmax(_fscores(mem, q, noise, adc_step), axis=1)


@jax.jit
def faulty_hopfield(mem, q, noise, adc_step, beta):
    w = jax.nn.softmax(beta * _fscores(mem, q, noise, adc_step), axis=-1)
    return jnp.argmax(jnp.matmul(jnp.matmul(w, mem, precision=HI), mem.T, precision=HI), axis=1)


def apply_fault(mem_i8, mode, sev, rng):
    m = mem_i8.astype(np.float32)
    if mode == "bitflip":
        m[rng.random(m.shape) < sev] *= -1
    elif mode == "stuck_zero":
        m[rng.random(m.shape) < sev] = 0.0
    elif mode == "stuck_pos":
        m[rng.random(m.shape) < sev] = 1.0
    elif mode == "dead_rows":
        m[rng.random(len(m)) < sev, :] = 0.0
    elif mode == "dead_cols":
        m[:, rng.random(m.shape[1]) < sev] = 0.0
    elif mode == "gauss_static":
        m += rng.normal(0.0, sev, m.shape).astype(np.float32)
    elif mode == "lognormal_gain":
        m *= np.exp(rng.normal(0.0, sev, m.shape)).astype(np.float32)
    return m


FAULTS = [("bitflip", [0.01, 0.05, 0.10, 0.20, 0.30]), ("stuck_zero", [0.05, 0.10, 0.25, 0.50, 0.75]), ("stuck_pos", [0.05, 0.10, 0.25, 0.50]),
          ("dead_rows", [0.01, 0.05, 0.10, 0.25]), ("dead_cols", [0.05, 0.10, 0.25, 0.50]), ("gauss_static", [0.25, 0.5, 1.0, 2.0, 4.0]),
          ("lognormal_gain", [0.1, 0.3, 0.5, 1.0]), ("read_noise", [0.005, 0.01, 0.02, 0.05]), ("adc_bits", [8, 6, 5, 4, 3, 2])]

if RUN_FAULTS:
    n_f, q_f = P["fault_n"], P["fault_q"]
    p_star = solve_noise_for_recall(D, n_f, 0.95)
    print(f"operating point: N={n_f}, D={D}, query flip probability {p_star:.4f} -> analytic fault-free recall {analytic_flat_recall(D, n_f, p_star):.4f}")
    METHODS = {"exact_flat": lambda m, q, nz, st: faulty_exact(m, q, nz, st), "hopfield_cleanup": lambda m, q, nz, st: faulty_hopfield(m, q, nz, st, BETA)}
    acc = {}
    for si in SEEDS:
        mem_i8 = bipolar_i8(n_f, D, rng_for(9, 0, si))
        tg, q_np = make_queries(mem_i8, q_f, p_star, rng_for(9, 1, si))
        q = to_f32(put(q_np))
        zero_noise, no_adc = put(np.zeros((q_f, n_f), np.float32)), jnp.float32(0.0)
        base_mem = put(mem_i8.astype(np.float32))
        for name, fn in METHODS.items():
            acc.setdefault(("baseline", 0.0, name), []).append(np.asarray(fn(base_mem, q, zero_noise, no_adc)) == tg)
        for mi, (mode, sevs) in enumerate(FAULTS):
            for sev in sevs:
                r = rng_for(9, 2, mi, int(round(sev * 1000)), si)
                mem, nz, st = base_mem, zero_noise, no_adc
                if mode == "read_noise":
                    nz = put(r.normal(0.0, sev, (q_f, n_f)).astype(np.float32))
                elif mode == "adc_bits":
                    st = jnp.float32(2.0 / (2 ** int(sev) - 1))
                else:
                    mem = put(apply_fault(mem_i8, mode, sev, r))
                for name, fn in METHODS.items():
                    acc.setdefault((mode, float(sev), name), []).append(np.asarray(fn(mem, q, nz, st)) == tg)
    ana = analytic_flat_recall(D, n_f, p_star)
    for (mode, sev, name), lst in acc.items():
        r1, lo, hi, trials = pooled_recall(lst)
        base = np.concatenate(acc[("baseline", 0.0, name)]); cur = np.concatenate(lst)
        b, c = int((base & ~cur).sum()), int((~base & cur).sum())                       # baseline-right/fault-wrong, and the reverse
        record("fault_injection", name, n=n_f, d=D, noise=p_star, fault_mode=mode, severity=sev, query_batch=q_f, trials=trials, recall_at_1=r1, recall_at_1_lo=lo,
               recall_at_1_hi=hi, mcnemar_p_vs_fault_free=mcnemar_exact(b, c), degraded_queries=b, improved_queries=c, analytic_fault_free=ana)
    fi = results_df("fault_injection")
    bl = fi[(fi.fault_mode == "baseline") & (fi.method == "exact_flat")].iloc[0]
    check("S9 fault-free exact recall matches the analytic model at the chosen operating point",
          abs(bl.recall_at_1 - ana) <= 4.5 * math.sqrt(ana * (1 - ana) / bl.trials) + 1.0 / bl.trials, f"measured {bl.recall_at_1:.4f}, analytic {ana:.4f}, trials {bl.trials}")
    # first severity with a significant degradation (paired McNemar p < 0.01 and more queries lost than gained)
    first = []
    for mode, sevs in FAULTS:
        order = sevs if mode != "adc_bits" else sevs                                  # listed from mild to harsh in both cases
        for name in METHODS:
            hit = None
            for sev in order:
                r = fi[(fi.fault_mode == mode) & (fi.severity == float(sev)) & (fi.method == name)].iloc[0]
                if r.mcnemar_p_vs_fault_free < 0.01 and r.degraded_queries > r.improved_queries:
                    hit = sev; break
            first.append({"fault_mode": mode, "method": name, "first_significant_degradation": hit if hit is not None else "none up to max severity"})
            record("fault_injection", name, n=n_f, d=D, fault_mode=mode, severity=-1.0, scenario="first_significant_degradation", query_batch=q_f,
                   first_significant_severity=hit if hit is not None else NAN)
    print("Fault severity at which recall first drops significantly below the fault-free value (paired McNemar p < 0.01):")
    display(pd.DataFrame(first).pivot(index="fault_mode", columns="method", values="first_significant_degradation").reindex([m for m, _ in FAULTS]))
    tbl = fi[fi.severity >= 0].copy()
    tbl["recall [95% CI]"] = [f"{r:.3f} [{lo:.3f},{hi:.3f}]" for r, lo, hi in zip(tbl.recall_at_1, tbl.recall_at_1_lo, tbl.recall_at_1_hi)]
    display(tbl.pivot_table(index=["fault_mode", "severity"], columns="method", values="recall_at_1", sort=False).round(3))

    fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharey=True)
    for ax, (mode, sevs) in zip(axes.ravel(), FAULTS):
        for name, mk in (("exact_flat", "o"), ("hopfield_cleanup", "s")):
            g = fi[(fi.fault_mode == mode) & (fi.method == name) & (fi.severity >= 0)].sort_values("severity")
            ax.errorbar(g.severity, g.recall_at_1, yerr=yerr(g.recall_at_1, g.recall_at_1_lo, g.recall_at_1_hi), marker=mk, ms=4, capsize=2, label=name)
        ax.axhline(ana, color="k", ls=":", lw=1); ax.set_title(mode, fontsize=10); ax.grid(alpha=.25)
        if mode == "adc_bits":
            ax.invert_xaxis()
    axes[0][0].legend(fontsize=7); fig.supxlabel("fault severity"); fig.supylabel("recall@1 (dotted: fault-free analytic)")
    plt.tight_layout(); savefig(fig, "s9_faults"); plt.show()
else:
    print("Fault injection disabled.")

## 10. Multi-device scaling

Everything above benchmarks **one** device. When the runtime exposes several (a Colab TPU has 8 cores), this section runs
the same exact top-1 kernel two ways with `jax.sharding`:

* **data parallel**: memory replicated, queries sharded across devices. Throughput should scale with the device count
  until the batch is too small to keep every device busy.
* **memory sharded**: memory split across devices, queries replicated, the argmax reduced across devices. This is how a
  store larger than one device's memory would be served; it adds a collective to every query.

Both are checked for exactness against a single-device reference (picked pattern attains the maximal score). With one device the
section is skipped. To exercise the logic on a CPU runtime, set `SIMULATE_CPU_DEVICES = 4` in the configuration cell and
restart the runtime; the timings are then meaningless (virtual devices share the same cores), but the correctness checks are real.

In [ ]:
#@title 10. Multi-device: data-parallel vs memory-sharded
clear("multidevice")
if RUN_MULTIDEVICE and len(DEVICES) > 1:
    from jax.sharding import Mesh, NamedSharding, PartitionSpec as PS
    nd = len(DEVICES)
    mesh = Mesh(np.array(DEVICES), ("d",))
    n_md = 2 ** int(math.log2(min(NMAX, 2 ** 16)))
    while n_md % nd:
        n_md //= 2
    q_md = max(nd, (min(max(P["Q"], 8 * nd), q_cap(n_md)) // nd) * nd)
    mem_i8 = bipolar_i8(n_md, D, rng_for(10, 0))
    tg, q_i8 = make_queries(mem_i8, q_md, 0.25, rng_for(10, 1))
    m32, q32 = to_f32(put(mem_i8)), to_f32(put(q_i8))
    rmax = ref_max_scores(q32[: min(q_md, 64)], m32)
    nvv = min(q_md, 64)
    rep, shard_rows = NamedSharding(mesh, PS()), NamedSharding(mesh, PS("d", None))

    def timing_row(name, fn, args, **kw):
        pred = np.asarray(fn(*args))
        st = bench(fn, *args, batch=q_md)
        return record("multidevice", name, n=n_md, d=D, query_batch=q_md, devices=nd, optimal_rate=optimal_rate(pred[:nvv], q_i8[:nvv], mem_i8, rmax[:nvv]),
                      recall_at_1=float(np.mean(pred == tg)), **st, **kw)

    r1 = timing_row("single_device", K_F32_HI, (q32, m32), speedup_vs_single=1.0, efficiency=1.0)
    dp_args = (jax.device_put(np.asarray(q32), shard_rows), jax.device_put(np.asarray(m32), rep))
    r2 = timing_row("data_parallel", K_F32_HI, dp_args)
    ms_args = (jax.device_put(np.asarray(q32), rep), jax.device_put(np.asarray(m32), shard_rows))
    r3 = timing_row("memory_sharded", K_F32_HI, ms_args)
    for row in (r2, r3):
        row["speedup_vs_single"] = r1["pipe_p50_ms"] / row["pipe_p50_ms"]
        row["efficiency"] = row["speedup_vs_single"] / nd
    show("multidevice", ["method", "devices", "n", "query_batch", "optimal_rate", "lat_p50_ms", "pipe_p50_ms", "qps", "speedup_vs_single", "efficiency"], sort=["method"])
    check("S10 data-parallel and memory-sharded results attain the maximal score (match the single-device reference)",
          bool(r2["optimal_rate"] == 1.0 and r3["optimal_rate"] == 1.0 and r1["optimal_rate"] == 1.0), f"{nd} devices, N={n_md}, Q={q_md}")
    if BACKEND == "cpu":
        print("NOTE: the devices are virtual CPU devices on the same cores; speedups above are not hardware scaling.")
else:
    print(f"Multi-device section skipped ({len(DEVICES)} local device(s); needs more than one and RUN_MULTIDEVICE).")

## 11. The NeuroStore prototype's core read kernel on the accelerator

The Python prototype in this repository reduces every read to one primitive: **sparse accumulate**. Take the `k` active
units, read their out-edge rows from the synapse lattice (CSR; each edge decays lazily as `w · (1-λ)^(epoch − edge_epoch)`),
and add them into a score array. The same primitive scores assemblies (unit → assembly inverted index) and spreads
activation. On a CPU/NumPy the CSR layout beat a dict-of-dicts by ~28× (see `REPORT.md`); this section asks the accelerator
question: **sparse scatter-add over a padded CSR vs one dense matrix product**, as the number of units `N` grows.

* `scatter_add`: gather `post`, `w`, `epoch` rows of the active units, apply the decay table, `at[].add` into `(B, N)`.
* `segment_sum`: the same updates through `jax.ops.segment_sum`.
* `dense_matmul`: materialise the effective weight matrix (`N × N`) once and multiply a multi-hot activation matrix. It
  becomes impossible when `N²` floats exceed device memory, and its work grows as `B·N²` against `B·k·R` for the sparse form.

All three are checked against a NumPy reference. The k-WTA selection step of the encoder (`top_k` vs `approx_max_k`) is timed too.

In [ ]:
#@title 11. Sparse accumulate (padded CSR) vs dense; k-WTA selection
clear("sparse_accumulate"); clear("kwta")


def make_graph(N, R, rng, epochs=30):
    """R distinct targets per unit (arithmetic progression with an odd stride, which is coprime with power-of-two N)."""
    start = rng.integers(0, N, size=(N, 1)).astype(np.int64)
    step = rng.integers(0, max(N // 2, 1), size=(N, 1)).astype(np.int64) * 2 + 1
    post = ((start + step * np.arange(R)[None, :]) % N).astype(np.int32)
    return post, rng.random((N, R)).astype(np.float32), rng.integers(0, epochs, (N, R)).astype(np.int32)


def make_actives(N, k, B, rng):
    act = np.stack([rng.choice(N, size=k, replace=False) for _ in range(B)]).astype(np.int32)
    return act, rng.random((B, k)).astype(np.float32) + 0.1


@partial(jax.jit, static_argnames=("N",))
def acc_scatter(post, w, ep, dtab, epoch, act, x, N):
    p = post[act]                                                       # (B, k, R)
    contrib = w[act] * dtab[epoch - ep[act]] * x[:, :, None]
    return jnp.zeros((act.shape[0], N), jnp.float32).at[jnp.arange(act.shape[0])[:, None, None], p].add(contrib)


@partial(jax.jit, static_argnames=("N",))
def acc_segment(post, w, ep, dtab, epoch, act, x, N):
    B = act.shape[0]
    p = post[act] + (jnp.arange(B, dtype=jnp.int32) * N)[:, None, None]
    contrib = w[act] * dtab[epoch - ep[act]] * x[:, :, None]
    return jax.ops.segment_sum(contrib.reshape(-1), p.reshape(-1), num_segments=B * N).reshape(B, N)


@partial(jax.jit, static_argnames=("N",))
def dense_weights(post, w, ep, dtab, epoch, N):
    eff = w * dtab[epoch - ep]
    return jnp.zeros((N, N), jnp.float32).at[jnp.arange(N)[:, None], post].add(eff)


@partial(jax.jit, static_argnames=("N",))
def acc_dense(Wd, act, x, N):
    A = jnp.zeros((act.shape[0], N), jnp.float32).at[jnp.arange(act.shape[0])[:, None], act].add(x)
    return jnp.matmul(A, Wd, precision=HI)


def np_reference(post, w, ep, dtab, epoch, act, x, N, rows=4):
    out = np.zeros((rows, N), np.float64)
    for b in range(rows):
        for j, u in enumerate(act[b]):
            np.add.at(out[b], post[u], w[u].astype(np.float64) * dtab[epoch - ep[u]] * x[b, j])
    return out


if RUN_SPARSE:
    R_, K_, EPOCH, LAMBDA = 128, 32, 30, 0.03
    dtab = (1.0 - LAMBDA) ** np.arange(EPOCH + 2, dtype=np.float32)
    bud = Budget(P["budget_s"])
    verified = []
    for N in P["sparse_N"]:
        if bud.expired():
            print(f"sparse sweep stopped before N={N}: budget used")
            break
        R = min(R_, N // 2)
        B = min(P["sparse_B"], max(1, int(q_cap(N, frac=4))))
        rng = rng_for(11, 0, N)
        post, w, ep = make_graph(N, R, rng, EPOCH); act, x = make_actives(N, min(K_, N // 4), B, rng)
        dpost, dw, dep, ddt, dact, dx = put(post), put(w), put(ep), put(dtab), put(act), put(x)
        sparse_bytes = N * R * (4 + 4 + 4)
        calls = {"scatter_add": lambda: acc_scatter(dpost, dw, dep, ddt, EPOCH, dact, dx, N), "segment_sum": lambda: acc_segment(dpost, dw, dep, ddt, EPOCH, dact, dx, N)}
        if fits(N * N, 4 * 2.2):                                       # dense matrix + one temporary
            Wd = dense_weights(dpost, dw, dep, ddt, EPOCH, N)
            calls["dense_matmul"] = lambda: acc_dense(Wd, dact, dx, N)
        ref = np_reference(post, w, ep, dtab, EPOCH, act, x, N)
        for name, call in calls.items():
            got = np.asarray(call())
            ok = bool(np.allclose(got[: ref.shape[0]], ref, rtol=2e-4, atol=2e-4 * max(1.0, float(np.abs(ref).max()))))
            verified.append(ok)
            st = bench(lambda call=call: call(), batch=B)
            flops = 2.0 * B * N * N if name == "dense_matmul" else float(B * min(K_, N // 4) * R)
            record("sparse_accumulate", name, n=N, d=D, query_batch=B, out_degree=R, active_units=min(K_, N // 4), matches_numpy_reference=ok,
                   weight_bytes=(N * N * 4) if name == "dense_matmul" else sparse_bytes, work_units=flops, **st)
        del calls
        gc.collect()
    sp = results_df("sparse_accumulate")
    check("S11 JAX sparse-accumulate kernels match the NumPy reference", bool(verified) and all(verified), f"{len(verified)} (kernel, N) cells")
    show("sparse_accumulate", ["method", "n", "query_batch", "out_degree", "matches_numpy_reference", "lat_p50_ms", "pipe_p50_ms", "us_per_query", "weight_bytes"], sort=["n", "method"])
    both = sp.pivot_table(index="n", columns="method", values="pipe_p50_ms")
    if {"scatter_add", "dense_matmul"} <= set(both.columns):
        both["dense_over_sparse"] = both["dense_matmul"] / both["scatter_add"]
        print("dense / sparse time ratio (>1: sparse wins; missing: dense matrix does not fit):")
        display(both.round(4))

    # k-WTA selection step of the encoder
    for N in P["sparse_N"][:4]:
        B = min(P["sparse_B"], max(1, int(q_cap(N, frac=4))))
        sc = put(rng_for(11, 1, N).random((B, N)).astype(np.float32))
        f_ex = jax.jit(lambda s: lax.top_k(s, 32)[1])
        f_ap = jax.jit(lambda s: lax.approx_max_k(s, 32, recall_target=0.95)[1])
        ex = np.asarray(f_ex(sc))
        for name, fn in (("top_k_exact", f_ex), ("approx_max_k_r0.95", f_ap)):
            try:
                got = np.asarray(fn(sc)); ov = float(np.mean([len(set(a) & set(b)) / 32 for a, b in zip(got, ex)]))
                record("kwta", name, n=N, d=D, query_batch=B, set_overlap_with_exact=ov, **bench(fn, sc, batch=B))
            except Exception as exc:
                print(f"{name} unavailable at N={N}: {type(exc).__name__}")
    show("kwta", ["method", "n", "query_batch", "set_overlap_with_exact", "lat_p50_ms", "pipe_p50_ms", "us_per_query"], sort=["n", "method"])

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for name in ("scatter_add", "segment_sum", "dense_matmul"):
        g = sp[sp.method == name].sort_values("n")
        if len(g):
            axes[0].plot(g.n, g.pipe_p50_ms, "o-", label=name); axes[1].plot(g.n, g.weight_bytes / 1e6, "o-", label=name)
    axes[0].set(xscale="log", yscale="log", xlabel="units N", ylabel="pipelined ms / batch", title="sparse accumulate"); axes[1].set(xscale="log", yscale="log", xlabel="units N", ylabel="weight storage (MB)", title="storage")
    for a in axes:
        a.grid(alpha=.25, which="both"); a.legend(fontsize=8)
    plt.tight_layout(); savefig(fig, "s11_sparse"); plt.show()
else:
    print("Sparse-accumulate study disabled.")

## 12. Self-checks, findings and claims ledger

Everything below is computed from the rows recorded above; nothing is typed in. A **FAIL** in the self-check table means an
invariant that must hold on every backend was violated on *this* runtime (a kernel not exact, a measured recall disagreeing
with theory, two implementations disagreeing). Treat such a runtime's timing results with suspicion until it is explained.

In [ ]:
#@title 12. Summary
df = results_df()
FINDINGS = []


def say(msg):
    FINDINGS.append(msg)
    print("-", msg)


print(f"{len(df)} result rows from {df.experiment.nunique()} experiments on {BACKEND}:{DEVICE_KIND} ({len(DEVICES)} device(s)), profile={PROFILE}, config={CONFIG_HASH}")
display(df.groupby("experiment").size().rename("rows").to_frame().T)
chk = pd.DataFrame(CHECKS)
if len(chk):
    display(chk.assign(status=np.where(chk.ok, "PASS", "FAIL"))[["status", "name", "detail"]])
    print(f"self-checks: {int(chk.ok.sum())}/{len(chk)} passed")
else:
    print("no self-checks were recorded")


def find(fn):
    try:
        fn()
    except (AttributeError, KeyError, IndexError, ValueError):       # a disabled or truncated section must not break the summary
        print(f"  (no data for: {fn.__name__[2:]})")


def f_saturation():
    ex = df[(df.experiment == "noisy_recall") & (df.method == "exact_flat")]
    sat = (ex.analytic_recall > 1 - 1e-9).mean()
    say(f"Noisy recall: {sat:.0%} of the (N, noise) cells have an analytic error probability below 1e-9, so only {1 - sat:.0%} can separate methods.")


def f_hopfield():
    ex = df[(df.experiment == "noisy_recall") & (df.method == "exact_flat")]
    hp = df[(df.experiment == "noisy_recall") & (df.method == "hopfield_cleanup")]
    for n in sorted(ex.n.unique()):
        e = ex[ex.n == n]; cell = e.iloc[(e.analytic_recall - 0.9).abs().argsort().iloc[0]]
        h = hp[(hp.n == n) & (hp.noise == cell.noise)].iloc[0]
        verdict = "no detectable difference" if h.mcnemar_p >= 0.05 else ("Hopfield worse" if h.paired_diff_vs_exact < 0 else "Hopfield better")
        say(f"Exact vs Hopfield at N={int(n)}, noise {cell.noise} (exact recall {cell.recall_at_1:.3f}): paired difference {h.paired_diff_vs_exact:+.3f} "
            f"[{h.paired_diff_lo:+.3f}, {h.paired_diff_hi:+.3f}], McNemar p={h.mcnemar_p:.3g}, {h.disagreement_with_exact:.1%} of queries differ -> {verdict}.")


def f_kernels():
    kv = df[df.experiment == "kernel_variants"]
    n = kv.n.max(); g = kv[kv.n == n].set_index("method")
    base = g.loc["f32_highest", "pipe_p50_ms"]
    parts = [f"{m} {base / g.loc[m, 'pipe_p50_ms']:.2f}x" for m in g.index if m != "f32_highest"]
    say(f"Kernels at N={int(n)}, Q={int(g.query_batch.iloc[0])}: speed relative to f32_highest: " + ", ".join(parts) +
        f"; popcount memory is {g.loc['packed_popcount', 'operand_bytes'] / g.loc['f32_highest', 'operand_bytes']:.3f}x of f32.")


def f_launch():
    sc = df[df.experiment == "scaling_N"]
    lb = (sc.lat_p50_ms < 5 * DISPATCH_FLOOR_MS)
    say(f"Scaling: {lb.mean():.0%} of N-sweep rows are within 5x of the dispatch floor ({DISPATCH_FLOOR_MS * 1e3:.0f} us); "
        f"best achieved {sc.gflops_effective.max():.0f} GFLOP/s and {sc.gbps_memory.max():.1f} GB/s at N up to {int(sc.n.max())}.")


def f_latency():
    ld = df[(df.experiment == "latency_dist") & (df.method == "f32_highest")]
    parts = [f"{r.path}: p50 {r.lat_p50_ms * 1e3:.0f} us, p99 {r.lat_p99_ms * 1e3:.0f} us" if r.lat_p99_ms == r.lat_p99_ms else f"{r.path}: p50 {r.lat_p50_ms * 1e3:.0f} us (p99 needs >=100 samples)" for r in ld.itertuples()]
    say("Single-request latency (Q=1, f32): " + "; ".join(parts) + ".")


def f_fly():
    fv = df[df.experiment == "fly_vs_simhash"]; n0 = fv.n.min(); nz = min(fv.noise.unique(), key=lambda x: abs(x - 0.30))
    g = fv[(fv.n == n0) & (fv.noise == nz)].set_index("method")
    say(f"FlyHash vs controls at N={int(n0)}, {nz:.0%} flips: " + ", ".join(f"{m} {g.loc[m, 'recall_at_1']:.3f}" for m in g.index) +
        f". Sizes: raw input {D / 8:.0f} B; v1 FlyHash code ideal {g[g.index.str.contains('v1')].ideal_bytes.iloc[0]:.0f} B.")


def f_vsa():
    vd = df[df.experiment == "vsa_decode"]; dd = vd.d.max()
    g = vd[vd.d == dd].sort_values("bundle_terms"); below = g[g.recall_at_1 < 0.9]
    ra = df[df.experiment == "vsa_records"]
    say(f"VSA at D={int(dd)}: decode accuracy stays >= 0.9 up to k={int(g[g.recall_at_1 >= 0.9].bundle_terms.max())} bundled pairs"
        + (f" and falls below it at k={int(below.bundle_terms.min())}" if len(below) else "") + f"; tuple-level record accuracy {ra.tuple_level_accuracy.min():.3f}-{ra.tuple_level_accuracy.max():.3f}.")


def f_sdm():
    sm = df[df.experiment == "sdm_summary"].dropna(subset=["memory_overhead_vs_flat_bitpacked"])
    say(f"SDM: best configuration reliably (>=90% exact) stores {int(sm.reliable_patterns_n90.max())} patterns; its memory per reliable pattern is "
        f"{sm.memory_overhead_vs_flat_bitpacked.min():.0f}x-{sm.memory_overhead_vs_flat_bitpacked.max():.0f}x that of a flat bit-packed store.")
    cap = df[(df.experiment == "sdm_capacity") & (df.noise > 0) & (df.read_iters == 1)]
    say(f"SDM: {int((cap.improves_over_input == False).sum())}/{len(cap)} noisy-cue cells return output with more bit errors than the cue.")  # noqa: E712


def f_ood():
    od = df[(df.experiment == "ood") & (df.scenario == "known25_vs_chimera")].set_index("method")
    say("OOD chimera family AUROC: " + ", ".join(f"{m} {od.loc[m, 'auroc']:.2f}" for m in od.index) + " (0.5 = chance).")


def f_faults():
    fi = df[(df.experiment == "fault_injection") & (df.scenario == "first_significant_degradation") & (df.method == "exact_flat")]
    say("Fault injection: first significant degradation of exact search (severity): " + ", ".join(f"{r.fault_mode} {r.first_significant_severity}" for r in fi.itertuples()) + ".")


def f_multi():
    md = df[df.experiment == "multidevice"].set_index("method")
    say(f"Multi-device ({int(md.devices.iloc[0])} devices): data-parallel speedup {md.loc['data_parallel', 'speedup_vs_single']:.2f}x, memory-sharded {md.loc['memory_sharded', 'speedup_vs_single']:.2f}x.")


def f_sparse():
    sp = df[df.experiment == "sparse_accumulate"].pivot_table(index="n", columns="method", values="pipe_p50_ms")
    parts = [f"N={int(n)}: sparse {r['scatter_add']:.3g} ms" + (f" vs dense {r['dense_matmul']:.3g} ms" if r.get('dense_matmul') == r.get('dense_matmul') else " (dense does not fit)") for n, r in sp.iterrows()]
    say("Sparse accumulate: " + "; ".join(parts) + ".")


print("\nFindings:")
for fn in (f_saturation, f_hopfield, f_kernels, f_launch, f_latency, f_fly, f_vsa, f_sdm, f_ood, f_faults, f_multi, f_sparse):
    find(fn)

# ---------------- claims ledger: what each result can and cannot support
LEDGER = []


def claim(text, status, evidence):
    LEDGER.append({"claim": text, "status": status, "evidence": evidence})


nr = df[df.experiment == "noisy_recall"]
if len(nr):
    hp = nr[nr.method == "hopfield_cleanup"]
    sig = hp[(hp.mcnemar_p < 0.05)]
    claim("Hopfield cleanup retrieves as well as exact flat search", "supported only where both are saturated; exact search is better in the informative cells" if len(sig) else "no detectable difference at this sample size",
          f"{len(sig)}/{len(hp)} (N, noise) cells have a significant paired difference (McNemar p<0.05); Section 1")
if CHECKS:
    ok = {c["name"]: c["ok"] for c in CHECKS}
    k_ok = [v for n_, v in ok.items() if n_.startswith("S3")]
    if k_ok:
        claim("bf16 / int8 / bit-packed kernels give the same exact-search answers as fp32 on +-1 data", "supported on this runtime" if all(k_ok) else "NOT supported on this runtime", "Section 3 optimal_rate == 1 for all exact kernels")
sc = df[df.experiment == "scaling_N"]
if len(sc):
    lb = float((sc.lat_p50_ms < 5 * DISPATCH_FLOOR_MS).mean())
    claim("Timings at v1's sizes (N <= 2048, Q <= 256) characterise accelerator throughput", "no" if BACKEND != "cpu" else "n/a on CPU", f"{lb:.0%} of N-sweep rows are launch-bound here; Section 4")
if len(df[df.experiment == "fly_vs_simhash"]):
    claim("FlyHash-style codes are more compact than the data they encode", "no for this projection", "the ideal k-of-m code size exceeds the D-bit raw input in the v1 configuration; Section 5")
if len(df[df.experiment == "sdm_summary"]):
    claim("SDM is memory-efficient relative to flat storage", "no in this setting", "bytes per reliable pattern vs a flat bit-packed store; Section 7")
if len(df[df.experiment == "fault_injection"]):
    claim("Fault injection shows robustness of the architecture", "only within the stated functional models and operating point", "Section 9 (no device physics)")
claim("Multi-core / full-pod scaling", "measured" if len(df[df.experiment == "multidevice"]) else "not measured (single device)", "Section 10")
claim("Energy per query, endurance, ADC/DAC overhead, crossbar area, FeNAND speed-ups", "not measured by this notebook", "no power or device model is present")
claim("Results reproduce the original report", "not established", "independent proxy implementation; seeds and sources not reconciled")
display(pd.DataFrame(LEDGER))

## 13. Export and cross-hardware comparison

Exports `results.csv` / `results.json` (every row carries run id, profile, base seed, config hash, backend, device kind
and JAX version), `hardware_metadata.json`, `config.json`, `checks.csv`, `findings.txt`, raw latency samples and all plots into one
ZIP. Colab's filesystem is ephemeral: **download the ZIP before disconnecting**.

The comparison cell merges several runs, refuses (with a warning) to compare runs whose profile or configuration hash differ,
and reports speed-ups relative to a chosen reference device plus the cross-device agreement of the accuracy metrics (identical
seeds should give identical recall on every backend apart from exact-score ties).

In [ ]:
#@title 13a. Export this runtime's result bundle
AUTO_DOWNLOAD = False #@param {type:"boolean"}

rdf = results_df()


def _jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if math.isnan(float(o)) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


rdf.to_csv(OUTPUT_DIR / "results.csv", index=False)
(OUTPUT_DIR / "results.json").write_text(rdf.to_json(orient="records", indent=1), encoding="utf-8")
(OUTPUT_DIR / "hardware_metadata.json").write_text(json.dumps(METADATA, indent=1, default=_jsonable), encoding="utf-8")
(OUTPUT_DIR / "config.json").write_text(json.dumps(CONFIG, indent=1, default=_jsonable), encoding="utf-8")
pd.DataFrame(CHECKS).to_csv(OUTPUT_DIR / "checks.csv", index=False)
pd.DataFrame(LEDGER).to_csv(OUTPUT_DIR / "claims_ledger.csv", index=False)
if LAT_SAMPLES:
    np.savez_compressed(OUTPUT_DIR / "latency_samples_ms.npz", **{f"{k[0]}__{k[1]}": v for k, v in LAT_SAMPLES.items()})
LIMITS = """NeuroStore accelerator benchmark lab v2: independent proxy implementation, not a reproduction of the original report's code.
One JAX device is benchmarked per section unless Section 10 says otherwise. Device-resident timings exclude transfers and compilation
(first_call_ms reports compilation). Batch throughput is not single-request latency. Percentiles are NaN unless enough samples exist.
FlyHash/VSA/SDM are algorithmic proxies; SDM is fixed-K. Fault injection is functional, not a calibrated device model.
Energy, endurance, ADC/DAC power, crossbar area and FeNAND speed-ups are NOT measured.
"""
(OUTPUT_DIR / "findings.txt").write_text(LIMITS + "\nFindings\n" + "\n".join("- " + f for f in FINDINGS) + "\n\nSelf-checks\n" +
                                         "\n".join(f"{'PASS' if c['ok'] else 'FAIL'} {c['name']} | {c['detail']}" for c in CHECKS) + "\n", encoding="utf-8")
ZIP_PATH = shutil.make_archive(str(OUTPUT_DIR.parent / f"NeuroStore_v2_{RUN_ID}_{BACKEND}_{PROFILE}"), "zip", root_dir=OUTPUT_DIR)
print(f"{len(rdf)} rows, {len(rdf.columns)} columns -> {OUTPUT_DIR}\nZIP: {ZIP_PATH}")
if AUTO_DOWNLOAD:
    from google.colab import files
    files.download(ZIP_PATH)
else:
    print("Set AUTO_DOWNLOAD=True and re-run this cell to download.")

In [ ]:
#@title 13b. Optional multi-runtime comparison
COMPARE_UPLOADS = False #@param {type:"boolean"}
COMPARE_GLOB = "" #@param {type:"string"}
REFERENCE_DEVICE = "" #@param {type:"string"}

if COMPARE_UPLOADS or COMPARE_GLOB:
    import glob, io
    frames = []
    if COMPARE_UPLOADS:
        from google.colab import files
        for fname, data in files.upload().items():
            if fname.lower().endswith(".csv"):
                frames.append(pd.read_csv(io.BytesIO(data)).assign(source_file=fname))
    for fname in sorted(glob.glob(COMPARE_GLOB)) if COMPARE_GLOB else []:
        frames.append(pd.read_csv(fname).assign(source_file=fname))
    if not frames:
        print("no CSV files found")
    else:
        comb = pd.concat(frames, ignore_index=True)
        comb["device"] = comb.backend.astype(str) + ":" + comb.device_kind.astype(str)
        for col in ("profile", "config_hash", "jax_version"):
            if col in comb and comb[col].nunique() > 1:
                print(f"WARNING: runs differ in {col}: {sorted(comb[col].astype(str).unique())}; timing comparisons may not be like for like.")
            if col not in comb:
                print(f"WARNING: {col} column missing (results from the v1 notebook?); comparability cannot be verified.")
        print("devices:", sorted(comb.device.unique()))
        ref = REFERENCE_DEVICE or sorted(comb.device.unique())[0]
        out = []
        kv = comb[comb.experiment == "kernel_variants"]
        if len(kv):
            nn_ = kv.groupby("device").n.max().min()
            t = kv[kv.n == nn_].pivot_table(index="method", columns="device", values="pipe_p50_ms")
            sp = (t[ref].values[:, None] / t) if ref in t else None
            print(f"\nkernel_variants at N={int(nn_)}: pipelined ms per call"); display(t.round(4))
            if sp is not None:
                print(f"speed-up relative to {ref} (>1 faster):"); display(sp.round(2))
            out.append(t)
        sc_ = comb[comb.experiment == "scaling_N"]
        if len(sc_):
            print("\nachieved GFLOP/s (max over N) by method and device"); display(sc_.pivot_table(index="method", columns="device", values="gflops_effective", aggfunc="max").round(1))
        ld = comb[(comb.experiment == "latency_dist") & (comb.path == "device_resident")] if "path" in comb else pd.DataFrame()
        if len(ld):
            print("\nsingle-request device-resident latency (us)"); display((ld.pivot_table(index="method", columns="device", values=["lat_p50_ms", "lat_p99_ms"]) * 1e3).round(1))
        nr_ = comb[(comb.experiment == "noisy_recall")]
        if len(nr_):
            acc_t = nr_.pivot_table(index=["method", "n", "noise"], columns="device", values="recall_at_1")
            spread = (acc_t.max(axis=1) - acc_t.min(axis=1)).max()
            print(f"\nmax cross-device spread of recall@1 over identical seeds: {spread:.4f} (0 expected apart from exact-score ties)")
        if "ok" in comb.columns:
            pass
        comb.to_csv("neurostore_cross_hardware_combined.csv", index=False)
        print("saved neurostore_cross_hardware_combined.csv")
else:
    print("To compare runtimes: download each ZIP, then set COMPARE_UPLOADS=True (Colab upload dialog) or COMPARE_GLOB='path/*/results.csv'.")